In [1]:
import sys
print(sys.executable)

c:\CricIQ\venv\Scripts\python.exe


In [2]:

import pandas as pd

file_path = r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"

df = pd.read_csv(file_path)

print("=" * 70)
print("CRICKETIQ — MATCH ID INTEGRITY AUDIT")
print("=" * 70)

print("\nTotal records:", len(df))

# Missing IDs
missing_ids = df["match_id"].isna()

print("\nMissing match IDs:", missing_ids.sum())
print("Non-missing match IDs:", df["match_id"].notna().sum())

print("\nMissing IDs by source:")
print(
    df.groupby("source")["match_id"]
    .apply(lambda x: x.isna().sum())
)

# Check duplicates only among non-missing IDs
valid_ids = df.loc[df["match_id"].notna(), "match_id"]

print("\nDuplicate non-null match IDs:", valid_ids.duplicated().sum())
print("Unique non-null match IDs:", valid_ids.nunique())

# Check whether missing IDs have identifiable match details
missing_id_records = df.loc[missing_ids]

print("\nMissing-ID records by result type:")
print(missing_id_records["result_type"].value_counts(dropna=False))

print("\nSample missing-ID records:")
display(
    missing_id_records[
        ["match_date", "team_1", "team_2", "venue",
         "winner", "result_type", "source"]
    ].head(15)
)

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


CRICKETIQ — MATCH ID INTEGRITY AUDIT

Total records: 2753

Missing match IDs: 153
Non-missing match IDs: 2600

Missing IDs by source:
source
Crickrida    153
Cricsheet      0
Name: match_id, dtype: int64

Duplicate non-null match IDs: 0
Unique non-null match IDs: 2600

Missing-ID records by result type:
result_type
Win          143
No result      9
Tie            1
Name: count, dtype: int64

Sample missing-ID records:


,match_date,team_1,team_2,venue,winner,result_type,source
807,2010-07-03,Afghanistan,Ireland,Hazelaarweg,Ireland,Win,Crickrida
1000,2012-02-10,Afghanistan,Pakistan,Sharjah Cricket Stadium,Pakistan,Win,Crickrida
1044,2012-07-03,Afghanistan,Ireland,Clontarf Cricket Club Ground,NaN,No result,Crickrida
1046,2012-07-05,Afghanistan,Ireland,Clontarf Cricket Club Ground,Ireland,Win,Crickrida
1059,2012-08-25,Afghanistan,Australia,Sharjah Cricket Stadium,Australia,Win,Crickrida
1224,2014-02-27,Afghanistan,Pakistan,Khan Shaheb Osman Ali Stadium,Pakistan,Win,Crickrida
1227,2014-03-01,Afghanistan,Bangladesh,Khan Shaheb Osman Ali Stadium,Afghanistan,Win,Crickrida
1230,2014-03-03,Afghanistan,Sri Lanka,Shere Bangla National Stadium,Sri Lanka,Win,Crickrida
1233,2014-03-05,Afghanistan,India,Shere Bangla National Stadium,India,Win,Crickrida
1250,2014-07-18,Afghanistan,Zimbabwe,Queens Sports Club,Zimbabwe,Win,Crickrida



AUDIT COMPLETE — NO FILES MODIFIED


In [3]:

import pandas as pd

file_path = r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"

matches = pd.read_csv(file_path)

# Preserve original row order and source match IDs
matches = matches.copy()

# Generate internal IDs
matches.insert(
    0,
    "cricketiq_match_id",
    [f"CRIQ_M_{i:06d}" for i in range(1, len(matches) + 1)]
)

print("=" * 70)
print("CRICKETIQ INTERNAL MATCH ID VALIDATION")
print("=" * 70)

print("\nDataset shape:", matches.shape)

print("\nFirst 10 internal IDs:")
display(
    matches[
        ["cricketiq_match_id", "match_id", "match_date",
         "team_1", "team_2", "source"]
    ].head(10)
)

print("\nInternal ID checks:")
print("Missing internal IDs:", matches["cricketiq_match_id"].isna().sum())
print("Duplicate internal IDs:", matches["cricketiq_match_id"].duplicated().sum())
print("Unique internal IDs:", matches["cricketiq_match_id"].nunique())

print("\nOriginal source IDs preserved:")
print(
    "Missing original IDs:",
    matches["match_id"].isna().sum()
)

print("\nSource distribution:")
print(matches["source"].value_counts())

assert matches["cricketiq_match_id"].notna().all()
assert matches["cricketiq_match_id"].is_unique
assert matches["match_id"].isna().sum() == 153

print("\nSUCCESS: Internal match IDs generated and validated.")
print("No files saved or modified.")


CRICKETIQ INTERNAL MATCH ID VALIDATION

Dataset shape: (2753, 41)

First 10 internal IDs:


,cricketiq_match_id,match_id,match_date,team_1,team_2,source
0,CRIQ_M_000001,66283.0,2002-06-27,England,Sri Lanka,Cricsheet
1,CRIQ_M_000002,66284.0,2002-06-29,England,India,Cricsheet
2,CRIQ_M_000003,64814.0,2002-12-29,New Zealand,India,Cricsheet
3,CRIQ_M_000004,64815.0,2003-01-01,India,New Zealand,Cricsheet
4,CRIQ_M_000005,64816.0,2003-01-04,India,New Zealand,Cricsheet
5,CRIQ_M_000006,64817.0,2003-01-08,New Zealand,India,Cricsheet
6,CRIQ_M_000007,65634.0,2003-01-09,Sri Lanka,Australia,Cricsheet
7,CRIQ_M_000008,65635.0,2003-01-11,Australia,England,Cricsheet
8,CRIQ_M_000009,65636.0,2003-01-13,Sri Lanka,England,Cricsheet
9,CRIQ_M_000010,64819.0,2003-01-14,India,New Zealand,Cricsheet



Internal ID checks:
Missing internal IDs: 0
Duplicate internal IDs: 0
Unique internal IDs: 2753

Original source IDs preserved:
Missing original IDs: 153

Source distribution:
source
Cricsheet    2600
Crickrida     153
Name: count, dtype: int64

SUCCESS: Internal match IDs generated and validated.
No files saved or modified.


In [4]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

match_file = Path(
    r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"
)

datasets = {
    "Cricsheet batting": "cricketiq_player_batting_canonical.csv",
    "Cricsheet bowling": "cricketiq_player_bowling_canonical.csv",
    "Cricsheet dismissals": "cricketiq_player_dismissals_canonical.csv",
    "Cricsheet dismissal bowler": "cricketiq_dismissal_bowler_canonical.csv",
    "Afghanistan batting": "afghanistan_player_batting_canonical.csv",
    "Afghanistan bowling": "afghanistan_player_bowling_canonical.csv",
}

matches = pd.read_csv(match_file)

print("=" * 80)
print("CRICKETIQ — PLAYER DATASET SCHEMA AND MATCH LINK AUDIT")
print("=" * 80)

print("\nMATCH DATASET")
print("Rows:", len(matches))
print("Unique non-null match IDs:", matches["match_id"].nunique())
print("Sources:")
print(matches["source"].value_counts())

for dataset_name, filename in datasets.items():

    path = player_dir / filename
    df = pd.read_csv(path)

    print("\n" + "-" * 80)
    print(dataset_name)
    print("-" * 80)

    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

    if "match_id" in df.columns:

        print("Missing match IDs:", df["match_id"].isna().sum())
        print("Unique match IDs:", df["match_id"].nunique())

        # Compare as strings to avoid integer/string dtype mismatches
        player_match_ids = set(
            df["match_id"].dropna().astype(str)
        )

        match_match_ids = set(
            matches["match_id"].dropna().astype(str)
        )

        matched_ids = player_match_ids & match_match_ids

        print("Match IDs found in match dataset:", len(matched_ids))
        print("Player IDs without match-dataset match:", 
              len(player_match_ids - match_match_ids))

    else:
        print("WARNING: No match_id column found.")

    print("\nSample records:")
    display(df.head(3))

print("\n" + "=" * 80)
print("SCHEMA INSPECTION COMPLETE — NO FILES MODIFIED")
print("=" * 80)


CRICKETIQ — PLAYER DATASET SCHEMA AND MATCH LINK AUDIT

MATCH DATASET
Rows: 2753
Unique non-null match IDs: 2600
Sources:
source
Cricsheet    2600
Crickrida     153
Name: count, dtype: int64

--------------------------------------------------------------------------------
Cricsheet batting
--------------------------------------------------------------------------------
Shape: (45432, 12)
Columns: ['match_id', 'innings_number', 'batting_team', 'batter', 'batter_id', 'runs', 'balls_faced', 'fours', 'sixes', 'strike_rate', 'cricketiq_player_id', 'canonical_player_name']
Missing match IDs: 0
Unique match IDs: 2600
Match IDs found in match dataset: 0
Player IDs without match-dataset match: 2600

Sample records:


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name
0,1000887,1,Australia,B Stanlake,6834d1f2,1,6,0,0,16.666667,CS_6834d1f2,B Stanlake
1,1000887,1,Australia,CA Lynn,45eda7c8,16,12,1,1,133.333333,CS_45eda7c8,CA Lynn
2,1000887,1,Australia,DA Warner,dcce6f09,7,18,1,0,38.888889,CRIQ_dcce6f09,David Warner



--------------------------------------------------------------------------------
Cricsheet bowling
--------------------------------------------------------------------------------
Shape: (31186, 13)
Columns: ['match_id', 'innings_number', 'bowler', 'bowler_id', 'runs_conceded', 'wides', 'no_balls', 'wickets', 'legal_balls', 'overs', 'economy', 'cricketiq_player_id', 'canonical_player_name']
Missing match IDs: 0
Unique match IDs: 2600
Match IDs found in match dataset: 0
Player IDs without match-dataset match: 2600

Sample records:


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets,legal_balls,overs,economy,cricketiq_player_id,canonical_player_name
0,1000887,1,Hasan Ali,2911de16,65,2,0,3,54,9.0,7.222222,CRIQ_2911de16,Hasan Ali
1,1000887,1,Imad Wasim,9cb8d7a6,35,0,0,2,60,10.0,3.500000,CRIQ_9cb8d7a6,Imad Wasim
2,1000887,1,Mohammad Amir,e174dadd,54,3,0,2,60,10.0,5.400000,CRIQ_e174dadd,Mohammad Amir



--------------------------------------------------------------------------------
Cricsheet dismissals
--------------------------------------------------------------------------------
Shape: (37711, 8)
Columns: ['match_id', 'innings_number', 'batting_team', 'player_out', 'player_out_id', 'dismissals', 'cricketiq_player_id', 'canonical_player_name']
Missing match IDs: 0
Unique match IDs: 2593
Match IDs found in match dataset: 0
Player IDs without match-dataset match: 2593

Sample records:


,match_id,innings_number,batting_team,player_out,player_out_id,dismissals,cricketiq_player_id,canonical_player_name
0,1000887,1,Australia,CA Lynn,45eda7c8,1,CS_45eda7c8,CA Lynn
1,1000887,1,Australia,DA Warner,dcce6f09,1,CRIQ_dcce6f09,David Warner
2,1000887,1,Australia,GJ Maxwell,b681e71e,1,CRIQ_b681e71e,Glenn Maxwell


... (3 outputs left out)

,player_name,dismissal,runs_scored,balls_faced,fours,sixes,innings_number,date,match_url,strike_rate,batting_team,cricketiq_player_id,canonical_player_name,legacy_cricketiq_player_id
0,Paul Stirling,bowled b Khaliq Dad,6,12,1,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,50.000000,Ireland,CRIQ_cf494ffe,Paul Stirling,CR_paulstirling
1,James Hall,caught Karim Sadiq b Shapoor Zadran,15,29,2,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,51.724138,Ireland,CR_jameshall,James Hall,CR_jameshall
2,Alex Cusack,caught Shabir Noori b Khaliq Dad,2,17,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,11.764706,Ireland,CRIQ_568e5a4a,Alex Cusack,CR_alexcusack



--------------------------------------------------------------------------------
Afghanistan bowling
--------------------------------------------------------------------------------
Shape: (1765, 18)
Columns: ['player_name', 'overs_display', 'maidens', 'runs_conceded', 'wickets', 'wides', 'no_balls', 'legal_balls', 'innings_number', 'date', 'match_url', 'economy', 'batting_team', 'bowling_team', 'opponent', 'cricketiq_player_id', 'canonical_player_name', 'legacy_cricketiq_player_id']

Sample records:


,player_name,overs_display,maidens,runs_conceded,wickets,wides,no_balls,legal_balls,innings_number,date,match_url,economy,batting_team,bowling_team,opponent,cricketiq_player_id,canonical_player_name,legacy_cricketiq_player_id
0,Shapoor Zadran,10.0,3,39,1,1,1,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.900,Ireland,Afghanistan,Ireland,CR_shapoorzadran,Shapoor Zadran,CR_shapoorzadran
1,Khaliq Dad,8.0,1,27,2,4,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.375,Ireland,Afghanistan,Ireland,CR_khaliqdad,Khaliq Dad,CR_khaliqdad
2,Hamid Hassan,8.0,0,44,3,0,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.500,Ireland,Afghanistan,Ireland,CR_hamidhassan,Hamid Hassan,CR_hamidhassan



SCHEMA INSPECTION COMPLETE — NO FILES MODIFIED


In [5]:

import pandas as pd
from pathlib import Path

match_file = Path(
    r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"
)

player_file = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
    r"\cricketiq_player_batting_canonical.csv"
)

matches = pd.read_csv(match_file)
batting = pd.read_csv(player_file)

print("=" * 75)
print("CRICKETIQ — CRICSHEET MATCH ID MISMATCH DIAGNOSTIC")
print("=" * 75)

print("\n1. COLUMN DTYPES")
print("Feature dataset match_id dtype:", matches["match_id"].dtype)
print("Player dataset match_id dtype:", batting["match_id"].dtype)

print("\n2. FIRST 15 MATCH IDs — FEATURE DATASET")
print(matches.loc[
    matches["source"] == "Cricsheet", "match_id"
].head(15).map(repr).to_list())

print("\n3. FIRST 15 MATCH IDs — PLAYER DATASET")
print(batting["match_id"].head(15).map(repr).to_list())

# Compare string representations
feature_ids = set(
    matches.loc[
        matches["source"] == "Cricsheet", "match_id"
    ].dropna().astype(str).str.strip()
)

player_ids = set(
    batting["match_id"].dropna().astype(str).str.strip()
)

print("\n4. STRING COMPARISON")
print("Feature unique IDs:", len(feature_ids))
print("Player unique IDs:", len(player_ids))
print("Exact overlaps:", len(feature_ids & player_ids))

# Try numeric normalization where possible
feature_numeric = pd.to_numeric(
    pd.Series(list(feature_ids)), errors="coerce"
).dropna()

player_numeric = pd.to_numeric(
    pd.Series(list(player_ids)), errors="coerce"
).dropna()

feature_numeric_ids = set(feature_numeric)
player_numeric_ids = set(player_numeric)

print("\n5. NUMERIC COMPARISON")
print("Feature numeric IDs:", len(feature_numeric_ids))
print("Player numeric IDs:", len(player_numeric_ids))
print("Numeric overlaps:", len(feature_numeric_ids & player_numeric_ids))

print("\n6. SAMPLE FEATURE MATCH RECORDS")
display(
    matches.loc[
        matches["source"] == "Cricsheet",
        ["match_id", "match_date", "team_1", "team_2", "venue"]
    ].head(10)
)

print("\n7. SAMPLE PLAYER MATCH RECORDS")
display(
    batting[
        ["match_id", "innings_number", "batting_team",
         "batter", "canonical_player_name"]
    ].head(10)
)

print("\nDIAGNOSTIC COMPLETE — NO FILES MODIFIED")


CRICKETIQ — CRICSHEET MATCH ID MISMATCH DIAGNOSTIC

1. COLUMN DTYPES
Feature dataset match_id dtype: float64
Player dataset match_id dtype: int64

2. FIRST 15 MATCH IDs — FEATURE DATASET
['66283.0', '66284.0', '64814.0', '64815.0', '64816.0', '64817.0', '65634.0', '65635.0', '65636.0', '64819.0', '65637.0', '65638.0', '65639.0', '65640.0', '65641.0']

3. FIRST 15 MATCH IDs — PLAYER DATASET
['1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887', '1000887']

4. STRING COMPARISON
Feature unique IDs: 2600
Player unique IDs: 2600
Exact overlaps: 0

5. NUMERIC COMPARISON
Feature numeric IDs: 2600
Player numeric IDs: 2600
Numeric overlaps: 2600

6. SAMPLE FEATURE MATCH RECORDS


,match_id,match_date,team_1,team_2,venue
0,66283.0,2002-06-27,England,Sri Lanka,"Trent Bridge, Nottingham"
1,66284.0,2002-06-29,England,India,"Lord's, London"
2,64814.0,2002-12-29,New Zealand,India,"McLean Park, Napier"
3,64815.0,2003-01-01,India,New Zealand,"Jade Stadium, Christchurch"
4,64816.0,2003-01-04,India,New Zealand,"Davies Park, Queenstown"
5,64817.0,2003-01-08,New Zealand,India,"Westpac Stadium, Wellington"
6,65634.0,2003-01-09,Sri Lanka,Australia,Sydney Cricket Ground
7,65635.0,2003-01-11,Australia,England,"Bellerive Oval, Hobart"
8,65636.0,2003-01-13,Sri Lanka,England,Sydney Cricket Ground
9,64819.0,2003-01-14,India,New Zealand,"Westpac Park, Hamilton"



7. SAMPLE PLAYER MATCH RECORDS


,match_id,innings_number,batting_team,batter,canonical_player_name
0,1000887,1,Australia,B Stanlake,B Stanlake
1,1000887,1,Australia,CA Lynn,CA Lynn
2,1000887,1,Australia,DA Warner,David Warner
3,1000887,1,Australia,GJ Maxwell,Glenn Maxwell
4,1000887,1,Australia,JP Faulkner,James Faulkner
5,1000887,1,Australia,MA Starc,Mitchell Starc
6,1000887,1,Australia,MR Marsh,Mitchell Marsh
7,1000887,1,Australia,MS Wade,Matthew Wade
8,1000887,1,Australia,PJ Cummins,Pat Cummins
9,1000887,1,Australia,SPD Smith,Steve Smith



DIAGNOSTIC COMPLETE — NO FILES MODIFIED


In [6]:

import pandas as pd
from pathlib import Path

base = Path(r"C:\CricIQ")

matches = pd.read_csv(
    base / "data/features/cricketiq_engineered_features.csv"
)

batting = pd.read_csv(
    base / "data/processed/player_data/identity_migration_20261123"
    / "cricketiq_player_batting_canonical.csv"
)

# Normalize IDs for comparison
matches["match_id_check"] = pd.to_numeric(
    matches["match_id"], errors="coerce"
).astype("Int64")

batting["match_id_check"] = pd.to_numeric(
    batting["match_id"], errors="coerce"
).astype("Int64")

cricsheet_matches = matches[
    matches["source"] == "Cricsheet"
].copy()

# Get the batting teams appearing in each Cricsheet match
batting_teams = (
    batting.groupby("match_id_check")["batting_team"]
    .agg(lambda x: sorted(set(x.dropna().astype(str))))
    .reset_index(name="player_batting_teams")
)

# Attach the match metadata
validation = cricsheet_matches.merge(
    batting_teams,
    on="match_id_check",
    how="left",
    validate="one_to_one"
)

# Check whether all observed batting teams belong to the match
def validate_teams(row):
    expected = {str(row["team_1"]), str(row["team_2"])}
    observed = set(row["player_batting_teams"] or [])
    return observed.issubset(expected) and len(observed) > 0

validation["team_check"] = validation.apply(
    validate_teams, axis=1
)

print("=" * 75)
print("CRICKETIQ — CRICSHEET MATCH AND TEAM VALIDATION")
print("=" * 75)

print("\nTotal Cricsheet matches:", len(validation))

print(
    "Matches with player batting records:",
    validation["player_batting_teams"].notna().sum()
)

print(
    "Matches with valid batting-team associations:",
    validation["team_check"].sum()
)

print(
    "Matches with team mismatches:",
    (~validation["team_check"]).sum()
)

print("\nTeam mismatch examples:")
display(
    validation.loc[
        ~validation["team_check"],
        ["match_id", "match_date", "team_1", "team_2",
         "player_batting_teams"]
    ].head(20)
)

print("\nSample validated matches:")
display(
    validation[
        ["match_id", "match_date", "team_1", "team_2",
         "player_batting_teams", "team_check"]
    ].head(10)
)

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


CRICKETIQ — CRICSHEET MATCH AND TEAM VALIDATION

Total Cricsheet matches: 2600
Matches with player batting records: 2600
Matches with valid batting-team associations: 2600
Matches with team mismatches: 0

Team mismatch examples:


,match_id,match_date,team_1,team_2,player_batting_teams



Sample validated matches:


,match_id,match_date,team_1,team_2,player_batting_teams,team_check
0,66283.0,2002-06-27,England,Sri Lanka,"[England, Sri Lanka]",True
1,66284.0,2002-06-29,England,India,"[England, India]",True
2,64814.0,2002-12-29,New Zealand,India,"[India, New Zealand]",True
3,64815.0,2003-01-01,India,New Zealand,"[India, New Zealand]",True
4,64816.0,2003-01-04,India,New Zealand,"[India, New Zealand]",True
5,64817.0,2003-01-08,New Zealand,India,"[India, New Zealand]",True
6,65634.0,2003-01-09,Sri Lanka,Australia,"[Australia, Sri Lanka]",True
7,65635.0,2003-01-11,Australia,England,"[Australia, England]",True
8,65636.0,2003-01-13,Sri Lanka,England,"[England, Sri Lanka]",True
9,64819.0,2003-01-14,India,New Zealand,"[India, New Zealand]",True



AUDIT COMPLETE — NO FILES MODIFIED


In [7]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

batting = pd.read_csv(
    player_dir / "afghanistan_player_batting_canonical.csv"
)

bowling = pd.read_csv(
    player_dir / "afghanistan_player_bowling_canonical.csv"
)

print("=" * 75)
print("CRICKETIQ — AFGHANISTAN MATCH URL AUDIT")
print("=" * 75)

for name, df in [
    ("Afghanistan batting", batting),
    ("Afghanistan bowling", bowling)
]:
    print(f"\n{name}")
    print("-" * 60)

    print("Rows:", len(df))
    print("Unique match URLs:", df["match_url"].nunique())
    print("Missing URLs:", df["match_url"].isna().sum())
    print("Unique dates:", df["date"].nunique())

    print("\nSample URLs:")
    display(
        df[["date", "match_url"]]
        .drop_duplicates()
        .head(10)
    )

# Build a URL-level fixture summary
batting_fixtures = (
    batting.groupby("match_url")
    .agg(
        date=("date", "first"),
        batting_teams=("batting_team",
                       lambda x: sorted(set(x.dropna()))),
        innings=("innings_number",
                 lambda x: sorted(set(x.dropna())))
    )
    .reset_index()
)

bowling_fixtures = (
    bowling.groupby("match_url")
    .agg(
        bowling_teams=("bowling_team",
                       lambda x: sorted(set(x.dropna()))),
        bowling_dates=("date",
                       lambda x: sorted(set(x.dropna())))
    )
    .reset_index()
)

fixture_summary = batting_fixtures.merge(
    bowling_fixtures,
    on="match_url",
    how="outer",
    indicator=True
)

print("\nURL-level fixture summary shape:", fixture_summary.shape)

print("\nBatting/bowling URL coverage:")
print(fixture_summary["_merge"].value_counts())

print("\nFirst 15 fixture summaries:")
display(fixture_summary.head(15))

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


CRICKETIQ — AFGHANISTAN MATCH URL AUDIT

Afghanistan batting
------------------------------------------------------------
Rows: 2719
Unique match URLs: 148
Missing URLs: 0
Unique dates: 148

Sample URLs:


,date,match_url
0,2010-07-03,https://crickrida.com/matches/afghanistan-irel...
21,2012-02-10,https://crickrida.com/matches/afghanistan-paki...
37,2012-07-05,https://crickrida.com/matches/afghanistan-irel...
59,2012-08-25,https://crickrida.com/matches/afghanistan-aust...
80,2014-02-27,https://crickrida.com/matches/afghanistan-paki...
101,2014-03-01,https://crickrida.com/matches/afghanistan-bang...
120,2014-03-03,https://crickrida.com/matches/afghanistan-sri-...
139,2014-03-05,https://crickrida.com/matches/afghanistan-indi...
154,2014-07-18,https://crickrida.com/matches/afghanistan-zimb...
170,2014-07-20,https://crickrida.com/matches/afghanistan-zimb...



Afghanistan bowling
------------------------------------------------------------
Rows: 1765
Unique match URLs: 148
Missing URLs: 0
Unique dates: 148

Sample URLs:


,date,match_url
0,2010-07-03,https://crickrida.com/matches/afghanistan-irel...
12,2012-02-10,https://crickrida.com/matches/afghanistan-paki...
25,2012-07-05,https://crickrida.com/matches/afghanistan-irel...
38,2012-08-25,https://crickrida.com/matches/afghanistan-aust...
52,2014-02-27,https://crickrida.com/matches/afghanistan-paki...
64,2014-03-01,https://crickrida.com/matches/afghanistan-bang...
78,2014-03-03,https://crickrida.com/matches/afghanistan-sri-...
91,2014-03-05,https://crickrida.com/matches/afghanistan-indi...
102,2014-07-18,https://crickrida.com/matches/afghanistan-zimb...
114,2014-07-20,https://crickrida.com/matches/afghanistan-zimb...



URL-level fixture summary shape: (148, 7)

Batting/bowling URL coverage:
_merge
both          148
left_only       0
right_only      0
Name: count, dtype: int64

First 15 fixture summaries:


,match_url,date,batting_teams,innings,bowling_teams,bowling_dates,_merge
0,https://crickrida.com/matches/afghanistan-aust...,2012-08-25,"[Afghanistan, Australia]","[1, 2]","[Afghanistan, Australia]",[2012-08-25],both
1,https://crickrida.com/matches/afghanistan-aust...,2015-03-04,"[Afghanistan, Australia]","[1, 2]","[Afghanistan, Australia]",[2015-03-04],both
2,https://crickrida.com/matches/afghanistan-aust...,2019-06-01,"[Afghanistan, Australia]","[1, 2]","[Afghanistan, Australia]",[2019-06-01],both
3,https://crickrida.com/matches/afghanistan-aust...,2023-11-07,"[Afghanistan, Australia]","[1, 2]","[Afghanistan, Australia]",[2023-11-07],both
4,https://crickrida.com/matches/afghanistan-aust...,2025-02-28,"[Afghanistan, Australia]","[1, 2]","[Afghanistan, Australia]",[2025-02-28],both
5,https://crickrida.com/matches/afghanistan-bang...,2014-03-01,"[Afghanistan, Bangladesh]","[1, 2]","[Afghanistan, Bangladesh]",[2014-03-01],both
6,https://crickrida.com/matches/afghanistan-bang...,2015-02-18,"[Afghanistan, Bangladesh]","[1, 2]","[Afghanistan, Bangladesh]",[2015-02-18],both
7,https://crickrida.com/matches/afghanistan-bang...,2016-09-25,"[Afghanistan, Bangladesh]","[1, 2]","[Afghanistan, Bangladesh]",[2016-09-25],both
8,https://crickrida.com/matches/afghanistan-bang...,2016-09-28,"[Afghanistan, Bangladesh]","[1, 2]","[Afghanistan, Bangladesh]",[2016-09-28],both
9,https://crickrida.com/matches/afghanistan-bang...,2016-10-01,"[Afghanistan, Bangladesh]","[1, 2]","[Afghanistan, Bangladesh]",[2016-10-01],both



AUDIT COMPLETE — NO FILES MODIFIED


In [8]:

import pandas as pd
from pathlib import Path

base = Path(r"C:\CricIQ")

match_df = pd.read_csv(
    base / "data/features/cricketiq_engineered_features.csv"
)

player_dir = (
    base / "data/processed/player_data/identity_migration_20261123"
)

batting = pd.read_csv(
    player_dir / "afghanistan_player_batting_canonical.csv"
)

bowling = pd.read_csv(
    player_dir / "afghanistan_player_bowling_canonical.csv"
)

# --------------------------------------------------
# 1. Prepare Crickrida Afghanistan match records
# --------------------------------------------------

crickrida_matches = match_df[
    match_df["source"].eq("Crickrida")
].copy()

crickrida_matches["match_date_key"] = pd.to_datetime(
    crickrida_matches["match_date"]
).dt.strftime("%Y-%m-%d")

afg_matches = crickrida_matches[
    crickrida_matches["team_1"].eq("Afghanistan")
    | crickrida_matches["team_2"].eq("Afghanistan")
].copy()

# --------------------------------------------------
# 2. Build fixture information from player records
# --------------------------------------------------

batting["date_key"] = pd.to_datetime(
    batting["date"]
).dt.strftime("%Y-%m-%d")

afg_fixtures = (
    batting.groupby(["date_key", "match_url"])
    .agg(
        batting_teams=(
            "batting_team",
            lambda x: sorted(set(x.dropna().astype(str)))
        ),
        innings_count=("innings_number", "nunique"),
        batting_records=("cricketiq_player_id", "size")
    )
    .reset_index()
)

# --------------------------------------------------
# 3. Compare date + team pairing
# --------------------------------------------------

def team_pair(teams):
    return tuple(sorted(str(t).strip() for t in teams))

afg_fixtures["team_pair"] = afg_fixtures[
    "batting_teams"
].apply(team_pair)

afg_matches["team_pair"] = afg_matches.apply(
    lambda r: team_pair([r["team_1"], r["team_2"]]),
    axis=1
)

# Check fixture key uniqueness
print("=" * 75)
print("AFGHANISTAN FIXTURE MATCHING VALIDATION")
print("=" * 75)

print("\nPlayer-derived fixtures:", len(afg_fixtures))
print("Crickrida Afghanistan matches:", len(afg_matches))

print("\nDuplicate player fixture keys:",
      afg_fixtures.duplicated(
          ["date_key", "team_pair"]
      ).sum())

print("Duplicate match fixture keys:",
      afg_matches.duplicated(
          ["match_date_key", "team_pair"]
      ).sum())

# --------------------------------------------------
# 4. Merge on date and both teams
# --------------------------------------------------

comparison = afg_fixtures.merge(
    afg_matches[
        ["match_date_key", "team_pair", "match_id",
         "venue", "winner", "result_type"]
    ],
    left_on=["date_key", "team_pair"],
    right_on=["match_date_key", "team_pair"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print("\nFixture comparison results:")
print(comparison["_merge"].value_counts())

print("\nMatched fixtures:",
      (comparison["_merge"] == "both").sum())

print("\nPlayer fixtures without match record:",
      (comparison["_merge"] == "left_only").sum())

print("\nMatch records without player fixture:",
      (comparison["_merge"] == "right_only").sum())

print("\nUnmatched player fixtures:")
display(
    comparison.loc[
        comparison["_merge"] == "left_only",
        ["date_key", "match_url", "batting_teams"]
    ].head(20)
)

print("\nUnmatched Crickrida matches:")
display(
    comparison.loc[
        comparison["_merge"] == "right_only",
        ["match_date_key", "team_pair", "match_id",
         "venue", "result_type"]
    ].head(20)
)

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


AFGHANISTAN FIXTURE MATCHING VALIDATION

Player-derived fixtures: 148
Crickrida Afghanistan matches: 153

Duplicate player fixture keys: 0
Duplicate match fixture keys: 0

Fixture comparison results:
_merge
both          146
right_only      7
left_only       2
Name: count, dtype: int64

Matched fixtures: 146

Player fixtures without match record: 2

Match records without player fixture: 7

Unmatched player fixtures:


,date_key,match_url,batting_teams
72,2019-03-02,https://crickrida.com/matches/afghanistan-irel...,[Afghanistan]
135,2024-12-17,https://crickrida.com/matches/afghanistan-zimb...,[Zimbabwe]



Unmatched Crickrida matches:


,match_date_key,team_pair,match_id,venue,result_type
2,2012-07-03,"(Afghanistan, Ireland)",NaN,Clontarf Cricket Club Ground,No result
30,2016-07-10,"(Afghanistan, Ireland)",NaN,Civil Service Cricket Club (Stormont),No result
50,2017-06-14,"(Afghanistan, West Indies)",NaN,Darren Sammy National Cricket Stadium,No result
73,2019-03-02,"(Afghanistan, Ireland)",NaN,Rajiv Gandhi International Cricket Stadium,No result
126,2024-03-09,"(Afghanistan, Ireland)",NaN,Sharjah Cricket Stadium,No result
134,2024-12-17,"(Afghanistan, Zimbabwe)",NaN,Harare Sports Club,No result
147,2026-08-05,"(Afghanistan, Ireland)",NaN,Bready Cricket Club,No result



AUDIT COMPLETE — NO FILES MODIFIED


In [9]:

import pandas as pd
from pathlib import Path

base = Path(r"C:\CricIQ")

match_df = pd.read_csv(
    base / "data/features/cricketiq_engineered_features.csv"
)

player_dir = (
    base / "data/processed/player_data/identity_migration_20261123"
)

batting = pd.read_csv(
    player_dir / "afghanistan_player_batting_canonical.csv"
)

bowling = pd.read_csv(
    player_dir / "afghanistan_player_bowling_canonical.csv"
)

# --------------------------------------------------
# 1. Prepare Afghanistan match records
# --------------------------------------------------

matches = match_df[
    (match_df["source"] == "Crickrida")
    & (
        (match_df["team_1"] == "Afghanistan")
        | (match_df["team_2"] == "Afghanistan")
    )
].copy()

matches["date_key"] = pd.to_datetime(
    matches["match_date"]
).dt.strftime("%Y-%m-%d")

matches["expected_teams"] = matches.apply(
    lambda r: sorted([r["team_1"], r["team_2"]]),
    axis=1
)

# --------------------------------------------------
# 2. Aggregate batting and bowling by date
# --------------------------------------------------

batting["date_key"] = pd.to_datetime(
    batting["date"]
).dt.strftime("%Y-%m-%d")

bowling["date_key"] = pd.to_datetime(
    bowling["date"]
).dt.strftime("%Y-%m-%d")

batting_summary = (
    batting.groupby("date_key")
    .agg(
        batting_teams=(
            "batting_team",
            lambda x: sorted(set(x.dropna().astype(str)))
        ),
        batting_records=("cricketiq_player_id", "size")
    )
    .reset_index()
)

bowling_summary = (
    bowling.groupby("date_key")
    .agg(
        bowling_teams=(
            "bowling_team",
            lambda x: sorted(set(x.dropna().astype(str)))
        ),
        bowling_records=("cricketiq_player_id", "size")
    )
    .reset_index()
)

# --------------------------------------------------
# 3. Combine player performance coverage
# --------------------------------------------------

player_summary = batting_summary.merge(
    bowling_summary,
    on="date_key",
    how="outer",
    validate="one_to_one"
)

player_summary["observed_teams"] = player_summary.apply(
    lambda r: sorted(
        set(r["batting_teams"] or [])
        | set(r["bowling_teams"] or [])
    ),
    axis=1
)

# --------------------------------------------------
# 4. Compare by date
# --------------------------------------------------

comparison = matches.merge(
    player_summary,
    on="date_key",
    how="outer",
    indicator=True,
    validate="one_to_one"
)

comparison["team_check"] = comparison.apply(
    lambda r: (
        set(r["observed_teams"]).issubset(
            set(r["expected_teams"])
        )
        if r["_merge"] == "both"
        else False
    ),
    axis=1
)

print("=" * 75)
print("AFGHANISTAN DATE-LEVEL MATCH RECONCILIATION")
print("=" * 75)

print("\nMatch dataset rows:", len(matches))
print("Unique player dates:", len(player_summary))

print("\nDate-level comparison:")
print(comparison["_merge"].value_counts())

print("\nMatches with valid observed team associations:",
      comparison["team_check"].sum())

print("\nMatches with team association issues:",
      ((comparison["_merge"] == "both")
       & (~comparison["team_check"])).sum())

print("\nNo-result matches and their player-data coverage:")

display(
    comparison.loc[
        (comparison["result_type"] == "No result")
        | (comparison["_merge"] != "both"),
        [
            "date_key", "team_1", "team_2",
            "result_type", "observed_teams",
            "batting_records", "bowling_records",
            "_merge", "team_check"
        ]
    ].sort_values("date_key")
)

print("\nDuplicate dates in match dataset:",
      matches["date_key"].duplicated().sum())

print("Duplicate dates in player summary:",
      player_summary["date_key"].duplicated().sum())

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


AFGHANISTAN DATE-LEVEL MATCH RECONCILIATION

Match dataset rows: 153
Unique player dates: 148

Date-level comparison:
_merge
both          148
left_only       5
right_only      0
Name: count, dtype: int64

Matches with valid observed team associations: 148

Matches with team association issues: 0

No-result matches and their player-data coverage:


,date_key,team_1,team_2,result_type,observed_teams,batting_records,bowling_records,_merge,team_check
2,2012-07-03,Afghanistan,Ireland,No result,NaN,NaN,NaN,left_only,False
30,2016-07-10,Afghanistan,Ireland,No result,NaN,NaN,NaN,left_only,False
50,2017-06-14,Afghanistan,West Indies,No result,NaN,NaN,NaN,left_only,False
72,2019-03-02,Afghanistan,Ireland,No result,"[Afghanistan, Ireland]",9.0,6.0,both,True
100,2022-11-27,Afghanistan,Sri Lanka,No result,"[Afghanistan, Sri Lanka]",13.0,8.0,both,True
125,2024-03-09,Afghanistan,Ireland,No result,NaN,NaN,NaN,left_only,False
133,2024-12-17,Afghanistan,Zimbabwe,No result,"[Afghanistan, Zimbabwe]",7.0,3.0,both,True
138,2025-02-28,Afghanistan,Australia,No result,"[Afghanistan, Australia]",14.0,10.0,both,True
145,2026-08-05,Afghanistan,Ireland,No result,NaN,NaN,NaN,left_only,False



Duplicate dates in match dataset: 0
Duplicate dates in player summary: 0

AUDIT COMPLETE — NO FILES MODIFIED


In [10]:

import pandas as pd
from pathlib import Path

base = Path(r"C:\CricIQ")

match_df = pd.read_csv(
    base / "data/features/cricketiq_engineered_features.csv"
)

player_dir = (
    base / "data/processed/player_data/identity_migration_20261123"
)

batting = pd.read_csv(
    player_dir / "afghanistan_player_batting_canonical.csv"
)

bowling = pd.read_csv(
    player_dir / "afghanistan_player_bowling_canonical.csv"
)

matches = match_df[
    (match_df["source"] == "Crickrida")
    & (
        (match_df["team_1"] == "Afghanistan")
        | (match_df["team_2"] == "Afghanistan")
    )
].copy()

matches["date_key"] = pd.to_datetime(
    matches["match_date"]
).dt.strftime("%Y-%m-%d")

batting["date_key"] = pd.to_datetime(
    batting["date"]
).dt.strftime("%Y-%m-%d")

bowling["date_key"] = pd.to_datetime(
    bowling["date"]
).dt.strftime("%Y-%m-%d")

batting_teams = (
    batting.groupby("date_key")["batting_team"]
    .agg(lambda x: set(x.dropna().astype(str)))
)

bowling_teams = (
    bowling.groupby("date_key")["bowling_team"]
    .agg(lambda x: set(x.dropna().astype(str)))
)

player_dates = sorted(set(batting["date_key"]) | set(bowling["date_key"]))

rows = []

for date in player_dates:
    bat_teams = batting_teams.get(date, set())
    bowl_teams = bowling_teams.get(date, set())

    observed = bat_teams | bowl_teams

    match = matches.loc[matches["date_key"] == date]

    if len(match) != 1:
        rows.append({
            "date": date,
            "expected_teams": None,
            "batting_teams": bat_teams,
            "bowling_teams": bowl_teams,
            "observed_teams": observed,
            "exact_match": False
        })
        continue

    expected = {
        str(match.iloc[0]["team_1"]),
        str(match.iloc[0]["team_2"])
    }

    rows.append({
        "date": date,
        "expected_teams": expected,
        "batting_teams": bat_teams,
        "bowling_teams": bowl_teams,
        "observed_teams": observed,
        "exact_match": observed == expected
    })

strict_audit = pd.DataFrame(rows)

print("=" * 75)
print("AFGHANISTAN EXACT TEAM-SET VALIDATION")
print("=" * 75)

print("\nPlayer fixtures:", len(strict_audit))
print("Exact team-set matches:", strict_audit["exact_match"].sum())
print("Team-set mismatches:", (~strict_audit["exact_match"]).sum())

print("\nMismatches:")
display(
    strict_audit.loc[~strict_audit["exact_match"]]
)

print("\nSample validated fixtures:")
display(strict_audit.head(10))

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


AFGHANISTAN EXACT TEAM-SET VALIDATION

Player fixtures: 148
Exact team-set matches: 148
Team-set mismatches: 0

Mismatches:


,date,expected_teams,batting_teams,bowling_teams,observed_teams,exact_match



Sample validated fixtures:


,date,expected_teams,batting_teams,bowling_teams,observed_teams,exact_match
0,2010-07-03,"{Afghanistan, Ireland}","{Afghanistan, Ireland}","{Afghanistan, Ireland}","{Afghanistan, Ireland}",True
1,2012-02-10,"{Afghanistan, Pakistan}","{Afghanistan, Pakistan}","{Afghanistan, Pakistan}","{Afghanistan, Pakistan}",True
2,2012-07-05,"{Afghanistan, Ireland}","{Afghanistan, Ireland}","{Afghanistan, Ireland}","{Afghanistan, Ireland}",True
3,2012-08-25,"{Afghanistan, Australia}","{Afghanistan, Australia}","{Afghanistan, Australia}","{Afghanistan, Australia}",True
4,2014-02-27,"{Afghanistan, Pakistan}","{Afghanistan, Pakistan}","{Afghanistan, Pakistan}","{Afghanistan, Pakistan}",True
5,2014-03-01,"{Afghanistan, Bangladesh}","{Afghanistan, Bangladesh}","{Afghanistan, Bangladesh}","{Afghanistan, Bangladesh}",True
6,2014-03-03,"{Afghanistan, Sri Lanka}","{Afghanistan, Sri Lanka}","{Afghanistan, Sri Lanka}","{Afghanistan, Sri Lanka}",True
7,2014-03-05,"{Afghanistan, India}","{Afghanistan, India}","{Afghanistan, India}","{Afghanistan, India}",True
8,2014-07-18,"{Zimbabwe, Afghanistan}","{Zimbabwe, Afghanistan}","{Zimbabwe, Afghanistan}","{Zimbabwe, Afghanistan}",True
9,2014-07-20,"{Zimbabwe, Afghanistan}","{Zimbabwe, Afghanistan}","{Zimbabwe, Afghanistan}","{Zimbabwe, Afghanistan}",True



AUDIT COMPLETE — NO FILES MODIFIED


In [11]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

files = {
    "Cricsheet batting": "cricketiq_player_batting_canonical.csv",
    "Afghanistan batting": "afghanistan_player_batting_canonical.csv"
}

for name, filename in files.items():

    df = pd.read_csv(player_dir / filename)

    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    numeric_cols = [
        c for c in [
            "runs", "runs_scored", "balls_faced",
            "fours", "sixes", "strike_rate"
        ] if c in df.columns
    ]

    print("\nNumeric summary:")
    display(df[numeric_cols].describe().round(2))

    print("\nZero-ball innings:")
    print((df["balls_faced"] == 0).sum())

    print("\nZero-run innings:")
    run_col = "runs" if "runs" in df.columns else "runs_scored"
    print((df[run_col] == 0).sum())

    print("\nMissing values:")
    print(df[numeric_cols].isna().sum())

    print("\nSample zero-ball records:")
    display(
        df.loc[
            df["balls_faced"] == 0
        ].head(5)
    )

print("\nBATTING DATA AUDIT COMPLETE — NO FILES MODIFIED")



CRICSHEET BATTING

Numeric summary:


,runs,balls_faced,fours,sixes,strike_rate
count,45432.00,45432.00,45432.00,45432.00,45423.00
mean,24.20,29.67,2.22,0.43,73.14
std,28.02,29.47,2.92,1.02,48.93
min,0.00,0.00,0.00,0.00,0.00
25%,4.00,8.00,0.00,0.00,42.86
50%,14.00,19.00,1.00,0.00,70.37
75%,35.00,42.00,3.00,0.00,100.00
max,264.00,173.00,33.00,16.00,600.00



Zero-ball innings:
9

Zero-run innings:
4330

Missing values:
runs           0
balls_faced    0
fours          0
sixes          0
strike_rate    9
dtype: int64

Sample zero-ball records:


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name
2778,1130737,1,Zimbabwe,SF Mire,3300d23c,0,0,0,0,NaN,CRIQ_3300d23c,Solomon Mire
5451,1168512,2,Bangladesh,Sabbir Rahman,7147f314,0,0,0,0,NaN,CRIQ_7147f314,Sabbir Rahman
10352,1323294,1,Bangladesh,Ebadat Hossain,cb5100f1,0,0,0,0,NaN,CRIQ_cb5100f1,Ebadot Hossain
15156,1432215,1,Pakistan,Naseem Shah,9c9af282,0,0,0,0,NaN,CRIQ_9c9af282,Naseem Shah
16053,1450701,2,Canada,Saad Bin Zafar,f28a60e0,0,0,0,0,NaN,CS_f28a60e0,Saad Bin Zafar



AFGHANISTAN BATTING

Numeric summary:


,runs_scored,balls_faced,fours,sixes,strike_rate
count,2719.00,2719.00,2719.00,2719.00,2691.00
mean,21.88,28.22,1.89,0.47,68.59
std,26.91,29.18,2.66,1.11,47.94
min,0.00,0.00,0.00,0.00,0.00
25%,3.00,7.00,0.00,0.00,38.46
50%,12.00,17.00,1.00,0.00,66.67
75%,31.00,40.00,3.00,0.00,91.67
max,210.00,151.00,22.00,17.00,400.00



Zero-ball innings:
28

Zero-run innings:
300

Missing values:
runs_scored     0
balls_faced     0
fours           0
sixes           0
strike_rate    28
dtype: int64

Sample zero-ball records:


,player_name,dismissal,runs_scored,balls_faced,fours,sixes,innings_number,date,match_url,strike_rate,batting_team,cricketiq_player_id,canonical_player_name,legacy_cricketiq_player_id
9,Rory McCann,run out Mohammad Nabi b Mohammad Nabi,0,0,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,NaN,Ireland,CRIQ_92c43fac,Rory McCann,CR_rorymccann
20,Shapoor Zadran,not out,0,0,0,0,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,NaN,Afghanistan,CR_shapoorzadran,Shapoor Zadran,CR_shapoorzadran
84,Misbah-ul-Haq,"run out Mohammad Shahzad, Najibullah Zadran b ...",0,0,0,0,1,2014-02-27,https://crickrida.com/matches/afghanistan-paki...,NaN,Pakistan,CRIQ_7eae4418,Misbah-ul-Haq,CRIQ_7eae4418
108,Mirwais Ashraf,not out,0,0,0,0,1,2014-03-01,https://crickrida.com/matches/afghanistan-bang...,NaN,Afghanistan,CR_mirwaisashraf,Mirwais Ashraf,CR_mirwaisashraf
138,Shapoor Zadran,not out,0,0,0,0,2,2014-03-03,https://crickrida.com/matches/afghanistan-sri-...,NaN,Afghanistan,CR_shapoorzadran,Shapoor Zadran,CR_shapoorzadran



BATTING DATA AUDIT COMPLETE — NO FILES MODIFIED


In [12]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

files = {
    "Cricsheet batting": "cricketiq_player_batting_canonical.csv",
    "Afghanistan batting": "afghanistan_player_batting_canonical.csv"
}

for name, filename in files.items():

    df = pd.read_csv(player_dir / filename)

    run_col = "runs" if "runs" in df.columns else "runs_scored"
    player_col = "cricketiq_player_id"
    name_col = "canonical_player_name"

    print("\n" + "=" * 75)
    print(name.upper(), "— IDENTITY AUDIT")
    print("=" * 75)

    print("\nTotal records:", len(df))
    print("Missing player IDs:", df[player_col].isna().sum())
    print("Missing canonical names:", df[name_col].isna().sum())

    print("\nUnique player IDs:", df[player_col].nunique())
    print("Unique canonical names:", df[name_col].nunique())

    print("\nRecords with missing player IDs:")
    display(
        df.loc[
            df[player_col].isna(),
            [name_col, run_col, "balls_faced",
             "innings_number"]
        ].head(15)
    )

    print("\nRecords with missing canonical names:")
    display(
        df.loc[df[name_col].isna()].head(10)
    )

    # Compare identity coverage across player ID and canonical name
    print("\nMissing ID but name present:",
          (df[player_col].isna() & df[name_col].notna()).sum())

    print("Both ID and name missing:",
          (df[player_col].isna() & df[name_col].isna()).sum())

print("\nIDENTITY AUDIT COMPLETE — NO FILES MODIFIED")



CRICSHEET BATTING — IDENTITY AUDIT

Total records: 45432
Missing player IDs: 0
Missing canonical names: 0

Unique player IDs: 1934
Unique canonical names: 1930

Records with missing player IDs:


,canonical_player_name,runs,balls_faced,innings_number



Records with missing canonical names:


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name



Missing ID but name present: 0
Both ID and name missing: 0

AFGHANISTAN BATTING — IDENTITY AUDIT

Total records: 2719
Missing player IDs: 0
Missing canonical names: 0

Unique player IDs: 403
Unique canonical names: 403

Records with missing player IDs:


,canonical_player_name,runs_scored,balls_faced,innings_number



Records with missing canonical names:


,player_name,dismissal,runs_scored,balls_faced,fours,sixes,innings_number,date,match_url,strike_rate,batting_team,cricketiq_player_id,canonical_player_name,legacy_cricketiq_player_id



Missing ID but name present: 0
Both ID and name missing: 0

IDENTITY AUDIT COMPLETE — NO FILES MODIFIED


In [13]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

files = {
    "Cricsheet bowling": "cricketiq_player_bowling_canonical.csv",
    "Afghanistan bowling": "afghanistan_player_bowling_canonical.csv"
}

for name, filename in files.items():

    df = pd.read_csv(player_dir / filename)

    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    numeric_cols = [
        c for c in [
            "runs_conceded", "wickets", "legal_balls",
            "economy", "overs", "maidens", "wides", "no_balls"
        ] if c in df.columns
    ]

    print("\nDataset shape:", df.shape)

    print("\nNumeric summary:")
    display(df[numeric_cols].describe().round(2))

    print("\nZero legal-ball records:")
    print((df["legal_balls"] == 0).sum())

    print("\nZero-wicket records:")
    print((df["wickets"] == 0).sum())

    print("\nMissing values:")
    print(df[numeric_cols].isna().sum())

    print("\nIdentity checks:")
    print("Missing player IDs:", df["cricketiq_player_id"].isna().sum())
    print("Missing canonical names:", df["canonical_player_name"].isna().sum())

    print("\nSample zero-legal-ball records:")
    display(
        df.loc[df["legal_balls"] == 0].head(5)
    )

print("\nBOWLING DATA AUDIT COMPLETE — NO FILES MODIFIED")



CRICSHEET BOWLING

Dataset shape: (31186, 13)

Numeric summary:


,runs_conceded,wickets,legal_balls,economy,overs,wides,no_balls
count,31186.00,31186.00,31186.00,31184.00,31186.00,31186.00,31186.00
mean,36.66,1.12,43.05,5.35,7.16,1.23,0.17
std,17.23,1.19,16.68,2.11,2.78,1.78,0.60
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,24.00,0.00,30.00,4.00,5.00,0.00,0.00
50%,36.00,1.00,48.00,5.10,8.00,1.00,0.00
75%,48.00,2.00,60.00,6.38,10.00,2.00,0.00
max,115.00,7.00,61.00,36.00,10.10,26.00,11.00



Zero legal-ball records:
2

Zero-wicket records:
11966

Missing values:
runs_conceded    0
wickets          0
legal_balls      0
economy          2
overs            0
wides            0
no_balls         0
dtype: int64

Identity checks:
Missing player IDs: 0
Missing canonical names: 0

Sample zero-legal-ball records:


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets,legal_balls,overs,economy,cricketiq_player_id,canonical_player_name
15529,249379,2,CO Obuya,5680f1ad,5,0,1,0,0,0.0,NaN,CS_5680f1ad,CO Obuya
28662,710305,1,Abdur Rehman,d094e7d1,8,0,3,0,0,0.0,NaN,CS_d094e7d1,Abdur Rehman



AFGHANISTAN BOWLING

Dataset shape: (1765, 18)

Numeric summary:


,runs_conceded,wickets,legal_balls,economy,maidens,wides,no_balls
count,1765.00,1765.00,1765.00,1764.00,1765.00,1765.00,1765.00
mean,35.04,1.21,43.37,5.03,0.36,1.00,0.09
std,16.55,1.24,16.29,1.95,0.66,1.28,0.35
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,23.00,0.00,30.00,3.71,0.00,0.00,0.00
50%,34.00,1.00,48.00,4.80,0.00,1.00,0.00
75%,46.00,2.00,60.00,6.00,1.00,2.00,0.00
max,110.00,7.00,60.00,19.00,4.00,8.00,4.00



Zero legal-ball records:
1

Zero-wicket records:
646

Missing values:
runs_conceded    0
wickets          0
legal_balls      0
economy          1
maidens          0
wides            0
no_balls         0
dtype: int64

Identity checks:
Missing player IDs: 0
Missing canonical names: 0

Sample zero-legal-ball records:


,player_name,overs_display,maidens,runs_conceded,wickets,wides,no_balls,legal_balls,innings_number,date,match_url,economy,batting_team,bowling_team,opponent,cricketiq_player_id,canonical_player_name,legacy_cricketiq_player_id
1515,Rahmanullah Gurbaz,0.0,0,1,0,1,0,0,2,2024-09-22,https://crickrida.com/matches/afghanistan-sout...,NaN,South Africa,Afghanistan,South Africa,CR_rahmanullahgurbaz,Rahmanullah Gurbaz,CR_rahmanullahgurbaz



BOWLING DATA AUDIT COMPLETE — NO FILES MODIFIED


In [14]:

import pandas as pd
import numpy as np
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

files = {
    "Cricsheet bowling": "cricketiq_player_bowling_canonical.csv",
    "Afghanistan bowling": "afghanistan_player_bowling_canonical.csv"
}

for name, filename in files.items():

    df = pd.read_csv(player_dir / filename)

    print("\n" + "=" * 75)
    print(name.upper(), "— FORMULA VALIDATION")
    print("=" * 75)

    calculated_economy = np.where(
        df["legal_balls"] > 0,
        df["runs_conceded"] * 6 / df["legal_balls"],
        np.nan
    )

    difference = np.abs(
        df["economy"].to_numpy(dtype=float)
        - calculated_economy
    )

    valid = np.isfinite(difference)

    print("Records:", len(df))
    print("Comparable economy records:", valid.sum())
    print("Economy mismatches (> 0.02):",
          (difference[valid] > 0.02).sum())

    print("Zero legal-ball records:",
          (df["legal_balls"] == 0).sum())

    # Check overs representation
    if "overs" in df.columns:
        print("\nCricsheet overs samples:")
        display(
            df[["overs", "legal_balls"]].head(10)
        )

    if "overs_display" in df.columns:
        print("\nAfghanistan overs samples:")
        display(
            df[["overs_display", "legal_balls"]].head(10)
        )

    print("\nSample formula mismatches:")
    mismatch_df = df.loc[
        valid & (difference > 0.02)
    ].copy()

    display(
        mismatch_df[
            ["runs_conceded", "legal_balls", "economy"]
        ].head(10)
    )

print("\nBOWLING FORMULA VALIDATION COMPLETE — NO FILES MODIFIED")



CRICSHEET BOWLING — FORMULA VALIDATION
Records: 31186
Comparable economy records: 31184
Economy mismatches (> 0.02): 0
Zero legal-ball records: 2

Cricsheet overs samples:


,overs,legal_balls
0,9.0,54
1,10.0,60
2,10.0,60
3,7.0,42
4,8.0,48
5,6.0,36
6,3.0,18
7,7.0,42
8,8.0,48
9,6.0,36



Sample formula mismatches:


,runs_conceded,legal_balls,economy



AFGHANISTAN BOWLING — FORMULA VALIDATION
Records: 1765
Comparable economy records: 1764
Economy mismatches (> 0.02): 0
Zero legal-ball records: 1

Afghanistan overs samples:


,overs_display,legal_balls
0,10.0,60
1,8.0,48
2,8.0,48
3,10.0,60
4,10.0,60
5,4.0,24
6,7.0,42
7,9.0,54
8,8.1,49
9,10.0,60



Sample formula mismatches:


,runs_conceded,legal_balls,economy



BOWLING FORMULA VALIDATION COMPLETE — NO FILES MODIFIED


In [15]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

datasets = {
    "Cricsheet batting": (
        "cricketiq_player_batting_canonical.csv",
        "match_id",
        "cricketiq_player_id"
    ),
    "Cricsheet bowling": (
        "cricketiq_player_bowling_canonical.csv",
        "match_id",
        "cricketiq_player_id"
    ),
    "Afghanistan batting": (
        "afghanistan_player_batting_canonical.csv",
        "date",
        "cricketiq_player_id"
    ),
    "Afghanistan bowling": (
        "afghanistan_player_bowling_canonical.csv",
        "date",
        "cricketiq_player_id"
    )
}

print("=" * 75)
print("CRICKETIQ — PLAYER CHRONOLOGICAL COVERAGE AUDIT")
print("=" * 75)

for name, (filename, date_col, player_col) in datasets.items():

    df = pd.read_csv(player_dir / filename)

    print("\n" + "-" * 75)
    print(name)
    print("-" * 75)

    if date_col == "date":
        dates = pd.to_datetime(df[date_col], errors="coerce")
    else:
        # Cricsheet match IDs connect to the match-level date table.
        dates = None

    print("Records:", len(df))
    print("Unique players:", df[player_col].nunique())

    if dates is not None:
        print("Earliest date:", dates.min())
        print("Latest date:", dates.max())
        print("Missing dates:", dates.isna().sum())

    else:
        print("Unique match IDs:", df[date_col].nunique())

    appearances = (
        df.groupby(player_col)
        .size()
        .describe()
    )

    print("\nRecords per player:")
    display(appearances.to_frame("records_per_player"))

    print("\nTop 10 players by records:")
    display(
        df.groupby([player_col, "canonical_player_name"])
        .size()
        .reset_index(name="records")
        .sort_values("records", ascending=False)
        .head(10)
    )

print("\nCHRONOLOGICAL COVERAGE AUDIT COMPLETE — NO FILES MODIFIED")


CRICKETIQ — PLAYER CHRONOLOGICAL COVERAGE AUDIT

---------------------------------------------------------------------------
Cricsheet batting
---------------------------------------------------------------------------
Records: 45432
Unique players: 1934
Unique match IDs: 2600

Records per player:


,records_per_player
count,1934.000000
mean,23.491210
std,36.997584
min,1.000000
25%,3.000000
50%,9.000000
75%,27.750000
max,302.000000



Top 10 players by records:


,cricketiq_player_id,canonical_player_name,records
295,CRIQ_ba607b88,Virat Kohli,302
154,CRIQ_6b71e6cf,Kumar Sangakkara,280
112,CRIQ_4a8a2e3b,MS Dhoni,276
167,CRIQ_740742ef,Rohit Sharma,276
330,CRIQ_d18f9182,Mahela Jayawardene,260
140,CRIQ_5bdcdb72,Tillakaratne Dilshan,259
261,CRIQ_a94e08ea,Mushfiqur Rahim,227
332,CRIQ_d2a6c0e6,Eoin Morgan,215
80,CRIQ_3b041a12,Tamim Iqbal,215
286,CRIQ_b61a3e1a,Ross Taylor,209



---------------------------------------------------------------------------
Cricsheet bowling
---------------------------------------------------------------------------
Records: 31186
Unique players: 1502
Unique match IDs: 2600

Records per player:


,records_per_player
count,1502.000000
mean,20.762983
std,30.926291
min,1.000000
25%,2.000000
50%,8.000000
75%,25.000000
max,208.000000


... (7 outputs left out)

,records_per_player
count,303.000000
mean,5.825083
std,12.891258
min,1.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,134.000000



Top 10 players by records:


,cricketiq_player_id,canonical_player_name,records
277,CR_mohammadnabi,Mohammad Nabi,134
290,CR_rashidkhan,Rashid Khan,110
279,CR_mujeeburrahman,Mujeeb Ur Rahman,67
267,CR_gulbadinnaib,Gulbadin Naib,66
264,CR_dawlatzadran,Dawlat Zadran,62
261,CR_azmatullahomarzai,Azmatullah Omarzai,48
266,CR_fazalhaqfarooqi,Fazalhaq Farooqi,44
292,CR_samiullahshinwari,Samiullah Shinwari,36
276,CR_mirwaisashraf,Mirwais Ashraf,25
259,CR_amirhamza,Amir Hamza,25



CHRONOLOGICAL COVERAGE AUDIT COMPLETE — NO FILES MODIFIED


In [16]:

import pandas as pd
import numpy as np
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

feature_file = Path(
    r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"
)

# Load match-level dataset
matches = pd.read_csv(feature_file)

# Normalize match IDs for reliable joining
matches["match_id_numeric"] = pd.to_numeric(
    matches["match_id"], errors="coerce"
)

matches["match_date"] = pd.to_datetime(
    matches["match_date"], errors="coerce"
)

# Select Cricsheet match records
cricsheet_matches = matches[
    matches["source"].str.lower() == "cricsheet"
].copy()

match_date_lookup = cricsheet_matches[
    ["match_id_numeric", "match_date"]
].dropna(subset=["match_id_numeric"])

print("=" * 75)
print("CRICKETIQ — CRICSHEET MATCH DATE RECONCILIATION")
print("=" * 75)

print("Cricsheet match records:", len(cricsheet_matches))
print("Unique Cricsheet match IDs:",
      cricsheet_matches["match_id_numeric"].nunique())

print("Duplicate match IDs in lookup:",
      match_date_lookup["match_id_numeric"].duplicated().sum())

# Check both player datasets
player_files = {
    "Cricsheet batting": "cricketiq_player_batting_canonical.csv",
    "Cricsheet bowling": "cricketiq_player_bowling_canonical.csv"
}

for name, filename in player_files.items():

    df = pd.read_csv(player_dir / filename)

    df["match_id_numeric"] = pd.to_numeric(
        df["match_id"], errors="coerce"
    )

    merged = df.merge(
        match_date_lookup,
        on="match_id_numeric",
        how="left",
        validate="many_to_one"
    )

    print("\n" + "-" * 75)
    print(name.upper())
    print("-" * 75)

    print("Player records:", len(merged))
    print("Records with matched dates:",
          merged["match_date"].notna().sum())
    print("Records without matched dates:",
          merged["match_date"].isna().sum())

    print("Unique players:", merged["cricketiq_player_id"].nunique())
    print("Unique matches:", merged["match_id_numeric"].nunique())

    print("Earliest match date:", merged["match_date"].min())
    print("Latest match date:", merged["match_date"].max())

print("\nMATCH DATE RECONCILIATION COMPLETE — NO FILES MODIFIED")


CRICKETIQ — CRICSHEET MATCH DATE RECONCILIATION
Cricsheet match records: 2600
Unique Cricsheet match IDs: 2600
Duplicate match IDs in lookup: 0

---------------------------------------------------------------------------
CRICSHEET BATTING
---------------------------------------------------------------------------
Player records: 45432
Records with matched dates: 45432
Records without matched dates: 0
Unique players: 1934
Unique matches: 2600
Earliest match date: 2002-06-27 00:00:00
Latest match date: 2026-10-25 00:00:00

---------------------------------------------------------------------------
CRICSHEET BOWLING
---------------------------------------------------------------------------
Player records: 31186
Records with matched dates: 31186
Records without matched dates: 0
Unique players: 1502
Unique matches: 2600
Earliest match date: 2002-06-27 00:00:00
Latest match date: 2026-10-25 00:00:00

MATCH DATE RECONCILIATION COMPLETE — NO FILES MODIFIED


In [17]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

files = {
    "Cricsheet batting": "cricketiq_player_batting_canonical.csv",
    "Cricsheet bowling": "cricketiq_player_bowling_canonical.csv",
    "Cricsheet dismissals": "cricketiq_player_dismissals_canonical.csv",
    "Afghanistan batting": "afghanistan_player_batting_canonical.csv",
    "Afghanistan bowling": "afghanistan_player_bowling_canonical.csv"
}

for name, filename in files.items():

    df = pd.read_csv(player_dir / filename)

    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nSample records:")
    display(df.head(3))

    if "innings_number" in df.columns:
        print("\nInnings-number distribution:")
        display(
            df["innings_number"]
            .value_counts(dropna=False)
            .sort_index()
            .to_frame("records")
        )

    if "dismissal" in df.columns:
        print("\nDismissal examples:")
        display(
            df["dismissal"]
            .dropna()
            .drop_duplicates()
            .head(15)
            .to_frame()
        )

# Check whether the Notebook 06 innings classification
# is still available in the current Jupyter kernel.
print("\n" + "=" * 75)
print("SUPER OVER CLASSIFICATION AVAILABILITY")
print("=" * 75)

print("innings_type_lookup available in kernel:",
      "innings_type_lookup" in globals())

print("\nINSPECTION COMPLETE — NO FILES MODIFIED")



CRICSHEET BATTING

Columns:
['match_id', 'innings_number', 'batting_team', 'batter', 'batter_id', 'runs', 'balls_faced', 'fours', 'sixes', 'strike_rate', 'cricketiq_player_id', 'canonical_player_name']

Sample records:


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name
0,1000887,1,Australia,B Stanlake,6834d1f2,1,6,0,0,16.666667,CS_6834d1f2,B Stanlake
1,1000887,1,Australia,CA Lynn,45eda7c8,16,12,1,1,133.333333,CS_45eda7c8,CA Lynn
2,1000887,1,Australia,DA Warner,dcce6f09,7,18,1,0,38.888889,CRIQ_dcce6f09,David Warner



Innings-number distribution:


,records
innings_number,
1,24403
2,21001
3,14
4,14



CRICSHEET BOWLING

Columns:
['match_id', 'innings_number', 'bowler', 'bowler_id', 'runs_conceded', 'wides', 'no_balls', 'wickets', 'legal_balls', 'overs', 'economy', 'cricketiq_player_id', 'canonical_player_name']

Sample records:


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets,legal_balls,overs,economy,cricketiq_player_id,canonical_player_name
0,1000887,1,Hasan Ali,2911de16,65,2,0,3,54,9.0,7.222222,CRIQ_2911de16,Hasan Ali
1,1000887,1,Imad Wasim,9cb8d7a6,35,0,0,2,60,10.0,3.500000,CRIQ_9cb8d7a6,Imad Wasim
2,1000887,1,Mohammad Amir,e174dadd,54,3,0,2,60,10.0,5.400000,CRIQ_e174dadd,Mohammad Amir


... (13 outputs left out)

,player_name,overs_display,maidens,runs_conceded,wickets,wides,no_balls,legal_balls,innings_number,date,match_url,economy,batting_team,bowling_team,opponent,cricketiq_player_id,canonical_player_name,legacy_cricketiq_player_id
0,Shapoor Zadran,10.0,3,39,1,1,1,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.900,Ireland,Afghanistan,Ireland,CR_shapoorzadran,Shapoor Zadran,CR_shapoorzadran
1,Khaliq Dad,8.0,1,27,2,4,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.375,Ireland,Afghanistan,Ireland,CR_khaliqdad,Khaliq Dad,CR_khaliqdad
2,Hamid Hassan,8.0,0,44,3,0,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.500,Ireland,Afghanistan,Ireland,CR_hamidhassan,Hamid Hassan,CR_hamidhassan



Innings-number distribution:


,records
innings_number,
1,906
2,859



SUPER OVER CLASSIFICATION AVAILABILITY
innings_type_lookup available in kernel: False

INSPECTION COMPLETE — NO FILES MODIFIED


In [18]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

feature_file = Path(
    r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"
)

# Load source tables
matches = pd.read_csv(feature_file)
matches["match_id_numeric"] = pd.to_numeric(
    matches["match_id"], errors="coerce"
)

batting = pd.read_csv(
    player_dir / "cricketiq_player_batting_canonical.csv"
)

bowling = pd.read_csv(
    player_dir / "cricketiq_player_bowling_canonical.csv"
)

cricsheet_dismissals = pd.read_csv(
    player_dir / "cricketiq_player_dismissals_canonical.csv"
)

afg_batting = pd.read_csv(
    player_dir / "afghanistan_player_batting_canonical.csv"
)

# --------------------------------------------------
# PART A: SUPER OVER AUDIT
# --------------------------------------------------

print("=" * 75)
print("PART A — CRICSHEET SUPER OVER AUDIT")
print("=" * 75)

batting["match_id_numeric"] = pd.to_numeric(
    batting["match_id"], errors="coerce"
)

bowling["match_id_numeric"] = pd.to_numeric(
    bowling["match_id"], errors="coerce"
)

super_batting = batting[batting["innings_number"] >= 3].copy()
super_bowling = bowling[bowling["innings_number"] >= 3].copy()

super_match_ids = sorted(
    set(super_batting["match_id_numeric"].dropna())
    | set(super_bowling["match_id_numeric"].dropna())
)

super_match_info = (
    matches[
        matches["match_id_numeric"].isin(super_match_ids)
    ][
        [
            "match_id_numeric",
            "match_date",
            "team_1",
            "team_2",
            "result_type",
            "winner"
        ]
    ]
    .drop_duplicates()
    .sort_values("match_date")
)

print("Super Over match IDs:", len(super_match_ids))
print("Super Over batting records:", len(super_batting))
print("Super Over bowling records:", len(super_bowling))

print("\nSuper Over match details:")
display(super_match_info)

print("\nResult type distribution:")
display(
    super_match_info["result_type"]
    .value_counts(dropna=False)
    .to_frame("matches")
)

print("\nSuper Over innings distribution — batting:")
display(
    super_batting["innings_number"]
    .value_counts()
    .sort_index()
    .to_frame("records")
)

print("\nSuper Over innings distribution — bowling:")
display(
    super_bowling["innings_number"]
    .value_counts()
    .sort_index()
    .to_frame("records")
)

# --------------------------------------------------
# PART B: CRICSHEET DISMISSAL AUDIT
# --------------------------------------------------

print("\n" + "=" * 75)
print("PART B — CRICSHEET DISMISSAL AUDIT")
print("=" * 75)

print("Dismissal records:", len(cricsheet_dismissals))
print("Unique dismissed players:",
      cricsheet_dismissals["cricketiq_player_id"].nunique())

print("\nDismissal count values:")
display(
    cricsheet_dismissals["dismissals"]
    .value_counts(dropna=False)
    .to_frame("records")
)

# --------------------------------------------------
# PART C: AFGHANISTAN DISMISSAL TEXT AUDIT
# --------------------------------------------------

print("\n" + "=" * 75)
print("PART C — AFGHANISTAN DISMISSAL TEXT AUDIT")
print("=" * 75)

afg_batting["dismissal_normalized"] = (
    afg_batting["dismissal"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Total batting records:", len(afg_batting))
print("Missing dismissal descriptions:",
      afg_batting["dismissal"].isna().sum())

print("\nAll distinct dismissal descriptions and counts:")
display(
    afg_batting["dismissal_normalized"]
    .value_counts(dropna=False)
    .to_frame("records")
)

print("\nPotential status descriptions:")
status_mask = afg_batting["dismissal_normalized"].str.contains(
    r"not out|retired|absent|did not bat|dnb",
    regex=True,
    na=False
)

display(
    afg_batting.loc[
        status_mask,
        ["player_name", "dismissal", "runs_scored", "balls_faced"]
    ]
    .drop_duplicates()
    .sort_values("dismissal")
)

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


PART A — CRICSHEET SUPER OVER AUDIT
Super Over match IDs: 6
Super Over batting records: 28
Super Over bowling records: 12

Super Over match details:


,match_id_numeric,match_date,team_1,team_2,result_type,winner
1910,1144530.0,2019-07-14,England,New Zealand,Tie,NaN
1981,1233463.0,2020-11-03,Zimbabwe,Pakistan,Tie,NaN
2308,1377763.0,2023-06-26,West Indies,Netherlands,Tie,NaN
2563,1474411.0,2025-03-09,Canada,Namibia,Tie,NaN
2579,1482566.0,2025-05-21,Oman,United States of America,Tie,NaN
2617,1505123.0,2025-10-21,Bangladesh,West Indies,Tie,NaN



Result type distribution:


,matches
result_type,
Tie,6



Super Over innings distribution — batting:


,records
innings_number,
3,14
4,14


... (5 outputs left out)

,records
dismissal_normalized,
not out,434
lbw b rashid khan,74
bowled b rashid khan,68
bowled b mujeeb ur rahman,31
lbw b mujeeb ur rahman,28
...,...
caught gavin hoey b jai moondra,1
caught suranga lakmal b lasith malinga,1
caught tafadzwa tsiga b brad evans,1



Potential status descriptions:


,player_name,dismissal,runs_scored,balls_faced
7,Trent Johnston,not out,42,28
1867,Mohammad Nabi,not out,27,28
1860,Lahiru Kumara,not out,4,5
1851,Dunith Wellalage,not out,31,21
1848,Charith Asalanka,not out,83,72
...,...,...,...,...
1366,Asghar Afghan,not out b George Dockrell,82,111
2567,Darwish Rasooli,not out b Gurnoor Brar,6,6
2015,Mehidy Hasan Miraz,not out b Mujeeb Ur Rahman,112,119
2495,Rahmat Shah,not out b Rishad Hossain,9,12



AUDIT COMPLETE — NO FILES MODIFIED


In [19]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

files = {
    "Cricsheet batting": "cricketiq_player_batting_canonical.csv",
    "Cricsheet bowling": "cricketiq_player_bowling_canonical.csv",
    "Afghanistan batting": "afghanistan_player_batting_canonical.csv",
    "Afghanistan bowling": "afghanistan_player_bowling_canonical.csv"
}

print("=" * 75)
print("CRICKETIQ — PLAYER-MATCH KEY VALIDATION")
print("=" * 75)

for name, filename in files.items():

    df = pd.read_csv(player_dir / filename)

    if name.startswith("Cricsheet"):

        df["match_key"] = (
            "CS_MATCH_" +
            pd.to_numeric(df["match_id"]).astype("Int64").astype(str)
        )

    else:

        df["match_date"] = pd.to_datetime(
            df["date"], errors="coerce"
        )

        if "batting_team" in df.columns:
            team_a = df["batting_team"]
            team_b = (
                df["batting_team"]
                if "bowling_team" not in df.columns
                else df["bowling_team"]
            )
        else:
            team_a = df["bowling_team"]
            team_b = df["opponent"]

        sorted_teams = pd.DataFrame({
            "team_a": team_a.astype(str),
            "team_b": team_b.astype(str)
        }).apply(
            lambda row: sorted([row["team_a"], row["team_b"]]),
            axis=1
        )

        df["match_key"] = (
            "CR_MATCH_" +
            df["match_date"].dt.strftime("%Y%m%d") +
            "_" +
            sorted_teams.str[0] +
            "_" +
            sorted_teams.str[1]
        )

    duplicate_mask = df.duplicated(
        subset=["match_key", "cricketiq_player_id"],
        keep=False
    )

    print("\n" + "-" * 75)
    print(name.upper())
    print("-" * 75)

    print("Records:", len(df))
    print("Unique match keys:", df["match_key"].nunique())
    print("Unique players:", df["cricketiq_player_id"].nunique())
    print("Duplicate player-match records:", duplicate_mask.sum())

    if duplicate_mask.any():
        display(
            df.loc[
                duplicate_mask,
                ["match_key", "cricketiq_player_id",
                 "canonical_player_name"]
            ].sort_values(
                ["match_key", "cricketiq_player_id"]
            ).head(20)
        )

print("\nPLAYER-MATCH KEY VALIDATION COMPLETE")
print("NO FILES MODIFIED")


CRICKETIQ — PLAYER-MATCH KEY VALIDATION

---------------------------------------------------------------------------
CRICSHEET BATTING
---------------------------------------------------------------------------
Records: 45432
Unique match keys: 2600
Unique players: 1934
Duplicate player-match records: 56


,match_key,cricketiq_player_id,canonical_player_name
4386,CS_MATCH_1144530,CRIQ_2be41edb,Martin Guptill
4403,CS_MATCH_1144530,CRIQ_2be41edb,Martin Guptill
4383,CS_MATCH_1144530,CRIQ_9219eff0,James Neesham
4402,CS_MATCH_1144530,CRIQ_9219eff0,James Neesham
4395,CS_MATCH_1144530,CRIQ_99b75528,Jos Buttler
4401,CS_MATCH_1144530,CRIQ_99b75528,Jos Buttler
4391,CS_MATCH_1144530,CRIQ_e087956b,Ben Stokes
4400,CS_MATCH_1144530,CRIQ_e087956b,Ben Stokes
7064,CS_MATCH_1233463,CRIQ_1777c020,Fakhar Zaman
7074,CS_MATCH_1233463,CRIQ_1777c020,Fakhar Zaman



---------------------------------------------------------------------------
CRICSHEET BOWLING
---------------------------------------------------------------------------
Records: 31186
Unique match keys: 2600
Unique players: 1502
Duplicate player-match records: 24


,match_key,cricketiq_player_id,canonical_player_name
2995,CS_MATCH_1144530,CRIQ_5574750c,Jofra Archer
3005,CS_MATCH_1144530,CRIQ_5574750c,Jofra Archer
3003,CS_MATCH_1144530,CRIQ_a818c1be,Trent Boult
3004,CS_MATCH_1144530,CRIQ_a818c1be,Trent Boult
4842,CS_MATCH_1233463,CRIQ_449f4e3c,Blessing Muzarabani
4848,CS_MATCH_1233463,CRIQ_449f4e3c,Blessing Muzarabani
4840,CS_MATCH_1233463,CRIQ_45a7e761,Shaheen Shah Afridi
4849,CS_MATCH_1233463,CRIQ_45a7e761,Shaheen Shah Afridi
8844,CS_MATCH_1377763,CRIQ_0f721006,Jason Holder
8848,CS_MATCH_1377763,CRIQ_0f721006,Jason Holder



---------------------------------------------------------------------------
AFGHANISTAN BATTING
---------------------------------------------------------------------------
Records: 2719
Unique match keys: 294
Unique players: 403
Duplicate player-match records: 0

---------------------------------------------------------------------------
AFGHANISTAN BOWLING
---------------------------------------------------------------------------
Records: 1765
Unique match keys: 148
Unique players: 303
Duplicate player-match records: 0

PLAYER-MATCH KEY VALIDATION COMPLETE
NO FILES MODIFIED


In [20]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

# Load the four performance tables
cs_bat = pd.read_csv(
    player_dir / "cricketiq_player_batting_canonical.csv"
)

cs_bowl = pd.read_csv(
    player_dir / "cricketiq_player_bowling_canonical.csv"
)

afg_bat = pd.read_csv(
    player_dir / "afghanistan_player_batting_canonical.csv"
)

afg_bowl = pd.read_csv(
    player_dir / "afghanistan_player_bowling_canonical.csv"
)

print("=" * 75)
print("PART A — CRICSHEET DUPLICATE INVESTIGATION")
print("=" * 75)

# Find players appearing more than once in a match
for name, df in [
    ("Cricsheet batting", cs_bat),
    ("Cricsheet bowling", cs_bowl)
]:

    df["match_id_numeric"] = pd.to_numeric(
        df["match_id"], errors="coerce"
    )

    duplicate_mask = df.duplicated(
        ["match_id_numeric", "cricketiq_player_id"],
        keep=False
    )

    duplicates = df.loc[duplicate_mask].copy()

    print("\n", name)
    print("Duplicate rows:", len(duplicates))
    print("Affected matches:",
          duplicates["match_id_numeric"].nunique())

    if len(duplicates):
        print("Duplicate innings distribution:")
        display(
            duplicates["innings_number"]
            .value_counts()
            .sort_index()
            .to_frame("records")
        )

        print("Duplicate records:")
        display(
            duplicates[
                [
                    "match_id_numeric",
                    "innings_number",
                    "cricketiq_player_id",
                    "canonical_player_name"
                ]
            ].sort_values(
                ["match_id_numeric", "cricketiq_player_id",
                 "innings_number"]
            ).head(20)
        )

    # Exclude Super Over innings
    regular = df[df["innings_number"].isin([1, 2])].copy()

    regular_duplicates = regular.duplicated(
        ["match_id_numeric", "cricketiq_player_id"],
        keep=False
    ).sum()

    print("Duplicate rows after excluding Super Overs:",
          regular_duplicates)


print("\n" + "=" * 75)
print("PART B — AFGHANISTAN FIXTURE KEY CORRECTION")
print("=" * 75)

# Normalize dates
afg_bat["date"] = pd.to_datetime(
    afg_bat["date"], errors="coerce"
)

afg_bowl["date"] = pd.to_datetime(
    afg_bowl["date"], errors="coerce"
)

# Build fixture map using both teams from bowling data
fixture_map = afg_bowl[
    ["date", "batting_team", "bowling_team"]
].drop_duplicates().copy()

fixture_map["team_pair"] = fixture_map.apply(
    lambda row: tuple(sorted([
        str(row["batting_team"]).strip(),
        str(row["bowling_team"]).strip()
    ])),
    axis=1
)

fixture_map["match_key"] = fixture_map.apply(
    lambda row:
        "CR_MATCH_" +
        row["date"].strftime("%Y%m%d") +
        "_" +
        row["team_pair"][0] +
        "_" +
        row["team_pair"][1],
    axis=1
)

# Map batting records using date and batting team
batting_key_map = fixture_map[
    ["date", "batting_team", "match_key"]
].drop_duplicates()

print("Fixture mapping rows:", len(batting_key_map))
print("Unique fixture keys:", batting_key_map["match_key"].nunique())

print("Duplicate date + batting-team mappings:",
      batting_key_map.duplicated(
          ["date", "batting_team"], keep=False
      ).sum())

afg_bat_corrected = afg_bat.merge(
    batting_key_map,
    on=["date", "batting_team"],
    how="left",
    validate="many_to_one"
)

print("\nAfghanistan batting records:",
      len(afg_bat_corrected))

print("Records with a matched fixture key:",
      afg_bat_corrected["match_key"].notna().sum())

print("Records without a fixture key:",
      afg_bat_corrected["match_key"].isna().sum())

print("Unique corrected match keys:",
      afg_bat_corrected["match_key"].nunique())

print("Duplicate player-match rows after correction:",
      afg_bat_corrected.duplicated(
          ["match_key", "cricketiq_player_id"],
          keep=False
      ).sum())

print("\n" + "=" * 75)
print("PART C — AFGHANISTAN BOWLING KEY CHECK")
print("=" * 75)

afg_bowl["match_key"] = afg_bowl.apply(
    lambda row:
        "CR_MATCH_" +
        row["date"].strftime("%Y%m%d") +
        "_" +
        "_".join(sorted([
            str(row["batting_team"]).strip(),
            str(row["bowling_team"]).strip()
        ])),
    axis=1
)

print("Bowling unique match keys:",
      afg_bowl["match_key"].nunique())

print("Bowling duplicate player-match rows:",
      afg_bowl.duplicated(
          ["match_key", "cricketiq_player_id"],
          keep=False
      ).sum())

print("\nKEY INVESTIGATION COMPLETE — NO FILES MODIFIED")


PART A — CRICSHEET DUPLICATE INVESTIGATION

 Cricsheet batting
Duplicate rows: 56
Affected matches: 6
Duplicate innings distribution:


,records
innings_number,
1,14
2,14
3,14
4,14


Duplicate records:


,match_id_numeric,innings_number,cricketiq_player_id,canonical_player_name
4386,1144530,1,CRIQ_2be41edb,Martin Guptill
4403,1144530,4,CRIQ_2be41edb,Martin Guptill
4383,1144530,1,CRIQ_9219eff0,James Neesham
4402,1144530,4,CRIQ_9219eff0,James Neesham
4395,1144530,2,CRIQ_99b75528,Jos Buttler
4401,1144530,3,CRIQ_99b75528,Jos Buttler
4391,1144530,2,CRIQ_e087956b,Ben Stokes
4400,1144530,3,CRIQ_e087956b,Ben Stokes
7064,1233463,2,CRIQ_1777c020,Fakhar Zaman
7074,1233463,3,CRIQ_1777c020,Fakhar Zaman


Duplicate rows after excluding Super Overs: 0

 Cricsheet bowling
Duplicate rows: 24
Affected matches: 6
Duplicate innings distribution:


,records
innings_number,
1,6
2,6
3,6
4,6


Duplicate records:


,match_id_numeric,innings_number,cricketiq_player_id,canonical_player_name
2995,1144530,1,CRIQ_5574750c,Jofra Archer
3005,1144530,4,CRIQ_5574750c,Jofra Archer
3003,1144530,2,CRIQ_a818c1be,Trent Boult
3004,1144530,3,CRIQ_a818c1be,Trent Boult
4842,1233463,2,CRIQ_449f4e3c,Blessing Muzarabani
4848,1233463,3,CRIQ_449f4e3c,Blessing Muzarabani
4840,1233463,1,CRIQ_45a7e761,Shaheen Shah Afridi
4849,1233463,4,CRIQ_45a7e761,Shaheen Shah Afridi
8844,1377763,2,CRIQ_0f721006,Jason Holder
8848,1377763,3,CRIQ_0f721006,Jason Holder


Duplicate rows after excluding Super Overs: 0

PART B — AFGHANISTAN FIXTURE KEY CORRECTION
Fixture mapping rows: 294
Unique fixture keys: 148
Duplicate date + batting-team mappings: 0

Afghanistan batting records: 2719
Records with a matched fixture key: 2719
Records without a fixture key: 0
Unique corrected match keys: 148
Duplicate player-match rows after correction: 0

PART C — AFGHANISTAN BOWLING KEY CHECK
Bowling unique match keys: 148
Bowling duplicate player-match rows: 0

KEY INVESTIGATION COMPLETE — NO FILES MODIFIED


In [21]:

import pandas as pd
import numpy as np
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

feature_file = Path(
    r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"
)

# ============================================================
# 1. LOAD SOURCE DATA
# ============================================================

matches = pd.read_csv(feature_file)

matches["match_id_numeric"] = pd.to_numeric(
    matches["match_id"], errors="coerce"
)

matches["match_date"] = pd.to_datetime(
    matches["match_date"], errors="coerce"
)

match_lookup = (
    matches[matches["source"].str.lower() == "cricsheet"]
    [
        [
            "match_id_numeric",
            "match_date",
            "team_1",
            "team_2"
        ]
    ]
    .drop_duplicates("match_id_numeric")
    .copy()
)

match_lookup["match_key"] = (
    "CS_MATCH_" +
    match_lookup["match_id_numeric"].astype("Int64").astype(str)
)

cs_bat = pd.read_csv(
    player_dir / "cricketiq_player_batting_canonical.csv"
)

cs_bowl = pd.read_csv(
    player_dir / "cricketiq_player_bowling_canonical.csv"
)

cs_dismissals = pd.read_csv(
    player_dir / "cricketiq_player_dismissals_canonical.csv"
)

afg_bat = pd.read_csv(
    player_dir / "afghanistan_player_batting_canonical.csv"
)

afg_bowl = pd.read_csv(
    player_dir / "afghanistan_player_bowling_canonical.csv"
)

# ============================================================
# 2. CRICSHEET BATTING + DISMISSALS
#    Standard ODI innings only: 1 and 2
# ============================================================

cs_bat["match_id_numeric"] = pd.to_numeric(
    cs_bat["match_id"], errors="coerce"
)

cs_bat = cs_bat[
    cs_bat["innings_number"].isin([1, 2])
].copy()

cs_dismissals["match_id_numeric"] = pd.to_numeric(
    cs_dismissals["match_id"], errors="coerce"
)

cs_dismissals = cs_dismissals[
    cs_dismissals["innings_number"].isin([1, 2])
].copy()

batting_keys = [
    "match_id_numeric",
    "innings_number",
    "cricketiq_player_id"
]

dismissal_keys = cs_dismissals[
    batting_keys +
    ["batting_team", "canonical_player_name"]
].drop_duplicates()

cs_bat = cs_bat.merge(
    dismissal_keys,
    on=batting_keys,
    how="outer",
    suffixes=("_bat", "_dismissal"),
    indicator=True,
    validate="one_to_one"
)

# Preserve the team and name from either source
cs_bat["batting_team"] = (
    cs_bat["batting_team_bat"]
    .combine_first(cs_bat["batting_team_dismissal"])
)

cs_bat["canonical_player_name"] = (
    cs_bat["canonical_player_name_bat"]
    .combine_first(cs_bat["canonical_player_name_dismissal"])
)

# A dismissal-only record represents a dismissed player
# who has no separate batting performance row in the source.
cs_bat["synthetic_batting_record"] = (
    cs_bat["_merge"] == "right_only"
)

cs_bat["is_out"] = cs_bat["_merge"].isin(
    ["both", "right_only"]
)

# Assign zero batting statistics only to those
# dismissal-only records, not to missing player performances.
synthetic_mask = cs_bat["synthetic_batting_record"]

for column in ["runs", "balls_faced", "fours", "sixes"]:
    cs_bat.loc[synthetic_mask, column] = 0

cs_bat = cs_bat.drop(
    columns=[
        "batting_team_bat",
        "batting_team_dismissal",
        "canonical_player_name_bat",
        "canonical_player_name_dismissal",
        "_merge"
    ]
)

# Attach match dates and teams
cs_bat = cs_bat.merge(
    match_lookup,
    on="match_id_numeric",
    how="left",
    validate="many_to_one"
)

# Determine opponent
cs_bat["opponent"] = np.where(
    cs_bat["batting_team"] == cs_bat["team_1"],
    cs_bat["team_2"],
    np.where(
        cs_bat["batting_team"] == cs_bat["team_2"],
        cs_bat["team_1"],
        np.nan
    )
)

cs_bat_std = pd.DataFrame({
    "match_key": cs_bat["match_key"],
    "source": "Cricsheet",
    "match_date": cs_bat["match_date"],
    "cricketiq_player_id": cs_bat["cricketiq_player_id"],
    "canonical_player_name": cs_bat["canonical_player_name"],
    "team": cs_bat["batting_team"],
    "opponent": cs_bat["opponent"],
    "batting_innings_number": cs_bat["innings_number"],
    "runs": cs_bat["runs"],
    "balls_faced": cs_bat["balls_faced"],
    "fours": cs_bat["fours"],
    "sixes": cs_bat["sixes"],
    "is_out": cs_bat["is_out"],
    "synthetic_batting_record": cs_bat["synthetic_batting_record"],
    "dismissal_description": np.nan
})

# ============================================================
# 3. CRICSHEET BOWLING
# ============================================================

cs_bowl["match_id_numeric"] = pd.to_numeric(
    cs_bowl["match_id"], errors="coerce"
)

cs_bowl = cs_bowl[
    cs_bowl["innings_number"].isin([1, 2])
].copy()

# Identify the batting team in each regular innings
innings_lookup = (
    cs_bat[
        [
            "match_id_numeric",
            "innings_number",
            "batting_team"
        ]
    ]
    .drop_duplicates()
)

cs_bowl = cs_bowl.merge(
    innings_lookup,
    on=["match_id_numeric", "innings_number"],
    how="left",
    validate="many_to_one"
)

cs_bowl = cs_bowl.merge(
    match_lookup,
    on="match_id_numeric",
    how="left",
    validate="many_to_one"
)

# The bowling team is the other team in the match
cs_bowl["team"] = np.where(
    cs_bowl["batting_team"] == cs_bowl["team_1"],
    cs_bowl["team_2"],
    np.where(
        cs_bowl["batting_team"] == cs_bowl["team_2"],
        cs_bowl["team_1"],
        np.nan
    )
)

cs_bowl_std = pd.DataFrame({
    "match_key": cs_bowl["match_key"],
    "source": "Cricsheet",
    "match_date": cs_bowl["match_date"],
    "cricketiq_player_id": cs_bowl["cricketiq_player_id"],
    "canonical_player_name": cs_bowl["canonical_player_name"],
    "team": cs_bowl["team"],
    "opponent": cs_bowl["batting_team"],
    "bowling_innings_number": cs_bowl["innings_number"],
    "runs_conceded": cs_bowl["runs_conceded"],
    "wickets": cs_bowl["wickets"],
    "legal_balls": cs_bowl["legal_balls"],
    "wides": cs_bowl["wides"],
    "no_balls": cs_bowl["no_balls"],
    "maidens": np.nan
})

# ============================================================
# 4. COMBINE CRICSHEET BATTING AND BOWLING
# ============================================================

cs_unified = cs_bat_std.merge(
    cs_bowl_std,
    on=["match_key", "cricketiq_player_id"],
    how="outer",
    suffixes=("_bat", "_bowl"),
    validate="one_to_one"
)

# Check metadata consistency before combining
cs_team_conflicts = (
    cs_unified["team_bat"].notna() &
    cs_unified["team_bowl"].notna() &
    (cs_unified["team_bat"] != cs_unified["team_bowl"])
)

cs_opponent_conflicts = (
    cs_unified["opponent_bat"].notna() &
    cs_unified["opponent_bowl"].notna() &
    (cs_unified["opponent_bat"] != cs_unified["opponent_bowl"])
)

cs_unified["source"] = "Cricsheet"

for column in ["match_date", "canonical_player_name", "team", "opponent"]:
    cs_unified[column] = (
        cs_unified[f"{column}_bat"]
        .combine_first(cs_unified[f"{column}_bowl"])
    )

cs_unified = cs_unified.drop(
    columns=[
        "source_bat", "source_bowl",
        "match_date_bat", "match_date_bowl",
        "canonical_player_name_bat",
        "canonical_player_name_bowl",
        "team_bat", "team_bowl",
        "opponent_bat", "opponent_bowl"
    ]
)

# ============================================================
# 5. AFGHANISTAN / CRICKRIDA FIXTURE MAPPING
# ============================================================

afg_bat["date"] = pd.to_datetime(
    afg_bat["date"], errors="coerce"
)

afg_bowl["date"] = pd.to_datetime(
    afg_bowl["date"], errors="coerce"
)

fixture_map = (
    afg_bowl[
        ["date", "batting_team", "bowling_team"]
    ]
    .drop_duplicates()
    .copy()
)

fixture_map["match_key"] = fixture_map.apply(
    lambda row:
        "CR_MATCH_" +
        row["date"].strftime("%Y%m%d") +
        "_" +
        "_".join(sorted([
            str(row["batting_team"]).strip(),
            str(row["bowling_team"]).strip()
        ])),
    axis=1
)

# Map batting records to their opponent
afg_bat = afg_bat.merge(
    fixture_map,
    on=["date", "batting_team"],
    how="left",
    validate="many_to_one"
)

# "not out" descriptions are not dismissals.
# Keep the original description for auditability.
afg_bat["is_out"] = ~(
    afg_bat["dismissal"]
    .fillna("")
    .astype(str)
    .str.contains(r"\bnot out\b", case=False, regex=True)
)

afg_bat_std = pd.DataFrame({
    "match_key": afg_bat["match_key"],
    "source": "Crickrida",
    "match_date": afg_bat["date"],
    "cricketiq_player_id": afg_bat["cricketiq_player_id"],
    "canonical_player_name": afg_bat["canonical_player_name"],
    "team": afg_bat["batting_team"],
    "opponent": afg_bat["bowling_team"],
    "batting_innings_number": afg_bat["innings_number"],
    "runs": afg_bat["runs_scored"],
    "balls_faced": afg_bat["balls_faced"],
    "fours": afg_bat["fours"],
    "sixes": afg_bat["sixes"],
    "is_out": afg_bat["is_out"],
    "synthetic_batting_record": False,
    "dismissal_description": afg_bat["dismissal"]
})

# ============================================================
# 6. AFGHANISTAN / CRICKRIDA BOWLING
# ============================================================

afg_bowl["match_key"] = afg_bowl.apply(
    lambda row:
        "CR_MATCH_" +
        row["date"].strftime("%Y%m%d") +
        "_" +
        "_".join(sorted([
            str(row["batting_team"]).strip(),
            str(row["bowling_team"]).strip()
        ])),
    axis=1
)

afg_bowl_std = pd.DataFrame({
    "match_key": afg_bowl["match_key"],
    "source": "Crickrida",
    "match_date": afg_bowl["date"],
    "cricketiq_player_id": afg_bowl["cricketiq_player_id"],
    "canonical_player_name": afg_bowl["canonical_player_name"],
    "team": afg_bowl["bowling_team"],
    "opponent": afg_bowl["batting_team"],
    "bowling_innings_number": afg_bowl["innings_number"],
    "runs_conceded": afg_bowl["runs_conceded"],
    "wickets": afg_bowl["wickets"],
    "legal_balls": afg_bowl["legal_balls"],
    "wides": afg_bowl["wides"],
    "no_balls": afg_bowl["no_balls"],
    "maidens": afg_bowl["maidens"]
})

# ============================================================
# 7. COMBINE AFGHANISTAN BATTING AND BOWLING
# ============================================================

afg_unified = afg_bat_std.merge(
    afg_bowl_std,
    on=["match_key", "cricketiq_player_id"],
    how="outer",
    suffixes=("_bat", "_bowl"),
    validate="one_to_one"
)

afg_team_conflicts = (
    afg_unified["team_bat"].notna() &
    afg_unified["team_bowl"].notna() &
    (afg_unified["team_bat"] != afg_unified["team_bowl"])
)

afg_opponent_conflicts = (
    afg_unified["opponent_bat"].notna() &
    afg_unified["opponent_bowl"].notna() &
    (afg_unified["opponent_bat"] != afg_unified["opponent_bowl"])
)

afg_unified["source"] = "Crickrida"

for column in ["match_date", "canonical_player_name", "team", "opponent"]:
    afg_unified[column] = (
        afg_unified[f"{column}_bat"]
        .combine_first(afg_unified[f"{column}_bowl"])
    )

afg_unified = afg_unified.drop(
    columns=[
        "source_bat", "source_bowl",
        "match_date_bat", "match_date_bowl",
        "canonical_player_name_bat",
        "canonical_player_name_bowl",
        "team_bat", "team_bowl",
        "opponent_bat", "opponent_bowl"
    ]
)

# ============================================================
# 8. CREATE FINAL UNIFIED PLAYER-MATCH DATASET
# ============================================================

unified_player_matches = pd.concat(
    [cs_unified, afg_unified],
    ignore_index=True,
    sort=False
)

unified_player_matches["is_super_over"] = False

unified_player_matches = unified_player_matches.sort_values(
    ["match_date", "match_key", "cricketiq_player_id"]
).reset_index(drop=True)

# Participation indicators
unified_player_matches["has_batting_record"] = (
    unified_player_matches["batting_innings_number"].notna()
)

unified_player_matches["has_bowling_record"] = (
    unified_player_matches["bowling_innings_number"].notna()
)

# ============================================================
# 9. VALIDATION REPORT
# ============================================================

print("=" * 75)
print("CRICKETIQ — UNIFIED PLAYER-MATCH DATASET")
print("=" * 75)

print("\nDataset shape:", unified_player_matches.shape)

print("\nSource distribution:")
display(
    unified_player_matches["source"]
    .value_counts()
    .to_frame("player_match_records")
)

print("\nParticipation breakdown:")
display(
    pd.Series({
        "Batting and bowling": (
            unified_player_matches["has_batting_record"] &
            unified_player_matches["has_bowling_record"]
        ).sum(),

        "Batting only": (
            unified_player_matches["has_batting_record"] &
            ~unified_player_matches["has_bowling_record"]
        ).sum(),

        "Bowling only": (
            ~unified_player_matches["has_batting_record"] &
            unified_player_matches["has_bowling_record"]
        ).sum()
    }).to_frame("records")
)

print("\nSynthetic Cricsheet batting records:",
      unified_player_matches["synthetic_batting_record"].fillna(False).sum())

print("\nDuplicate player-match keys:",
      unified_player_matches.duplicated(
          ["match_key", "cricketiq_player_id"],
          keep=False
      ).sum())

print("\nMissing match keys:",
      unified_player_matches["match_key"].isna().sum())

print("Missing match dates:",
      unified_player_matches["match_date"].isna().sum())

print("Missing player IDs:",
      unified_player_matches["cricketiq_player_id"].isna().sum())

print("\nTeam conflicts:")
print("Cricsheet:", cs_team_conflicts.sum())
print("Crickrida:", afg_team_conflicts.sum())

print("\nOpponent conflicts:")
print("Cricsheet:", cs_opponent_conflicts.sum())
print("Crickrida:", afg_opponent_conflicts.sum())

print("\nDate range:")
print("Earliest:", unified_player_matches["match_date"].min())
print("Latest:", unified_player_matches["match_date"].max())

print("\nFinal columns:")
print(unified_player_matches.columns.tolist())

print("\nSample records:")
display(unified_player_matches.head(10))

print("\nUNIFIED DATASET BUILD COMPLETE — NOT SAVED TO DISK")


CRICKETIQ — UNIFIED PLAYER-MATCH DATASET

Dataset shape: (58319, 25)

Source distribution:


,player_match_records
source,
Cricsheet,55148
Crickrida,3171



Participation breakdown:


,records
Batting and bowling,22822
Batting only,25380
Bowling only,10117



Synthetic Cricsheet batting records: 79

Duplicate player-match keys: 0

Missing match keys: 0
Missing match dates: 0
Missing player IDs: 0

Team conflicts:
Cricsheet: 0
Crickrida: 0

Opponent conflicts:
Cricsheet: 0
Crickrida: 0

Date range:
Earliest: 2002-06-27 00:00:00
Latest: 2026-10-25 00:00:00

Final columns:
['match_key', 'cricketiq_player_id', 'batting_innings_number', 'runs', 'balls_faced', 'fours', 'sixes', 'is_out', 'synthetic_batting_record', 'dismissal_description', 'bowling_innings_number', 'runs_conceded', 'wickets', 'legal_balls', 'wides', 'no_balls', 'maidens', 'source', 'match_date', 'canonical_player_name', 'team', 'opponent', 'is_super_over', 'has_batting_record', 'has_bowling_record']

Sample records:


C:\Users\ASUS\AppData\Local\Temp\ipykernel_12484\2585199797.py:498: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  unified_player_matches["synthetic_batting_record"].fillna(False).sum())


,match_key,cricketiq_player_id,batting_innings_number,runs,balls_faced,fours,sixes,is_out,synthetic_batting_record,dismissal_description,...,no_balls,maidens,source,match_date,canonical_player_name,team,opponent,is_super_over,has_batting_record,has_bowling_record
0,CS_MATCH_66283,CRIQ_6b71e6cf,2.0,22.0,28.0,2.0,0.0,True,False,NaN,...,NaN,NaN,Cricsheet,2002-06-27,Kumar Sangakkara,Sri Lanka,England,False,True,False
1,CS_MATCH_66283,CRIQ_d18f9182,2.0,29.0,37.0,3.0,0.0,True,False,NaN,...,NaN,NaN,Cricsheet,2002-06-27,Mahela Jayawardene,Sri Lanka,England,False,True,False
2,CS_MATCH_66283,CS_22860288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,NaN,Cricsheet,2002-06-27,RJ Kirtley,England,Sri Lanka,False,False,True
3,CS_MATCH_66283,CS_342d8ade,2.0,5.0,13.0,0.0,0.0,False,False,NaN,...,1.0,NaN,Cricsheet,2002-06-27,CRD Fernando,Sri Lanka,England,False,True,True
4,CS_MATCH_66283,CS_434e36b1,2.0,1.0,4.0,0.0,0.0,True,False,NaN,...,0.0,NaN,Cricsheet,2002-06-27,UDU Chandana,Sri Lanka,England,False,True,True
5,CS_MATCH_66283,CS_53fcbbfb,1.0,18.0,31.0,1.0,0.0,True,False,NaN,...,NaN,NaN,Cricsheet,2002-06-27,GP Thorpe,England,Sri Lanka,False,True,False
6,CS_MATCH_66283,CS_8a686d30,2.0,47.0,58.0,7.0,0.0,True,False,NaN,...,NaN,NaN,Cricsheet,2002-06-27,MS Atapattu,Sri Lanka,England,False,True,False
7,CS_MATCH_66283,CS_8c82ae9a,2.0,15.0,22.0,0.0,0.0,False,False,NaN,...,NaN,NaN,Cricsheet,2002-06-27,MN Nawaz,Sri Lanka,England,False,True,False
8,CS_MATCH_66283,CS_a386e91b,1.0,10.0,3.0,2.0,0.0,False,False,NaN,...,0.0,NaN,Cricsheet,2002-06-27,PD Collingwood,England,Sri Lanka,False,True,True
9,CS_MATCH_66283,CS_a4af4849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,Cricsheet,2002-06-27,AF Giles,England,Sri Lanka,False,False,True



UNIFIED DATASET BUILD COMPLETE — NOT SAVED TO DISK


In [22]:

# ============================================================
# CORRECT PARTICIPATION FLAGS
# ============================================================

# Treat missing synthetic flags as False
synthetic = unified_player_matches[
    "synthetic_batting_record"
].eq(True)

# A batting performance exists in the original source
unified_player_matches["has_batting_record"] = (
    unified_player_matches["batting_innings_number"].notna()
    & ~synthetic
)

# Includes actual batting records AND dismissal-only records
unified_player_matches["has_batting_involvement"] = (
    unified_player_matches["batting_innings_number"].notna()
)

unified_player_matches["has_bowling_record"] = (
    unified_player_matches["bowling_innings_number"].notna()
)

unified_player_matches["synthetic_batting_record"] = synthetic

unified_player_matches["is_super_over"] = False

# ============================================================
# VALIDATION
# ============================================================

print("=" * 75)
print("CRICKETIQ — PARTICIPATION FLAG VALIDATION")
print("=" * 75)

print("\nTotal unified player-match records:",
      len(unified_player_matches))

print("\nBatting and bowling participation:")
display(
    pd.DataFrame({
        "count": [
            unified_player_matches["has_batting_record"].sum(),
            unified_player_matches["has_batting_involvement"].sum(),
            unified_player_matches["has_bowling_record"].sum(),
            synthetic.sum()
        ]
    }, index=[
        "Actual batting records",
        "Batting involvement including synthetic dismissals",
        "Bowling records",
        "Synthetic dismissal-only records"
    ])
)

# Check synthetic records
synthetic_rows = unified_player_matches.loc[synthetic]

print("\nSynthetic dismissal validation:")
print("Synthetic records:", len(synthetic_rows))
print("All marked out:",
      synthetic_rows["is_out"].eq(True).all())
print("All have zero runs:",
      synthetic_rows["runs"].eq(0).all())
print("All have zero balls:",
      synthetic_rows["balls_faced"].eq(0).all())

# Check batting dismissal status
batting_rows = unified_player_matches[
    unified_player_matches["has_batting_involvement"]
]

print("\nBatting dismissal status:")
print("Batting involvement records:", len(batting_rows))
print("Missing is_out values:", batting_rows["is_out"].isna().sum())

# Check final key integrity
print("\nFinal integrity:")
print("Duplicate player-match keys:",
      unified_player_matches.duplicated(
          ["match_key", "cricketiq_player_id"]
      ).sum())

print("Missing match keys:",
      unified_player_matches["match_key"].isna().sum())

print("Missing player IDs:",
      unified_player_matches["cricketiq_player_id"].isna().sum())

print("\nPARTICIPATION FLAG VALIDATION COMPLETE")
print("DATASET STILL IN MEMORY — NOT SAVED")


CRICKETIQ — PARTICIPATION FLAG VALIDATION

Total unified player-match records: 58319

Batting and bowling participation:


,count
Actual batting records,48123
Batting involvement including synthetic dismissals,48202
Bowling records,32939
Synthetic dismissal-only records,79



Synthetic dismissal validation:
Synthetic records: 79
All marked out: True
All have zero runs: True
All have zero balls: True

Batting dismissal status:
Batting involvement records: 48202
Missing is_out values: 0

Final integrity:
Duplicate player-match keys: 0
Missing match keys: 0
Missing player IDs: 0

PARTICIPATION FLAG VALIDATION COMPLETE
DATASET STILL IN MEMORY — NOT SAVED


In [23]:

import pandas as pd
from pathlib import Path

player_dir = Path(
    r"C:\CricIQ\data\processed\player_data\identity_migration_20261123"
)

# Reload the original published dismissal data
cs_dismissals_check = pd.read_csv(
    player_dir / "cricketiq_player_dismissals_canonical.csv"
)

# Standard ODI innings only
cs_dismissals_check = cs_dismissals_check[
    cs_dismissals_check["innings_number"].isin([1, 2])
].copy()

# Select Cricsheet records from the unified dataset
cs_unified_check = unified_player_matches[
    unified_player_matches["source"] == "Cricsheet"
].copy()

cs_batting_check = cs_unified_check[
    cs_unified_check["has_batting_involvement"]
].copy()

# Separate actual batting records and synthetic dismissals
cs_actual_batting = cs_batting_check[
    ~cs_batting_check["synthetic_batting_record"]
]

cs_synthetic = cs_batting_check[
    cs_batting_check["synthetic_batting_record"]
]

print("=" * 75)
print("CRICKETIQ — DISMISSAL RECONCILIATION")
print("=" * 75)

print("\nCRICSHEET")
print("-" * 75)

print("Source dismissal records:", len(cs_dismissals_check))

print("Unified batting involvement:",
      len(cs_batting_check))

print("Actual batting records:",
      len(cs_actual_batting))

print("Synthetic dismissal records:",
      len(cs_synthetic))

print("Dismissals marked out in unified data:",
      cs_batting_check["is_out"].eq(True).sum())

print("Actual batting records marked out:",
      cs_actual_batting["is_out"].eq(True).sum())

print("Source dismissal key duplicates:",
      cs_dismissals_check.duplicated(
          ["match_id", "innings_number", "cricketiq_player_id"]
      ).sum())

print("\nAFGHANISTAN")
print("-" * 75)

afg_unified_check = unified_player_matches[
    unified_player_matches["source"] == "Crickrida"
].copy()

afg_batting_check = afg_unified_check[
    afg_unified_check["has_batting_involvement"]
].copy()

print("Afghanistan batting records:", len(afg_batting_check))
print("Marked out:",
      afg_batting_check["is_out"].eq(True).sum())

print("Marked not out:",
      afg_batting_check["is_out"].eq(False).sum())

print("Missing dismissal flags:",
      afg_batting_check["is_out"].isna().sum())

print("\nFINAL CHECKS")
print("-" * 75)

print(
    "Cricsheet dismissal reconciliation:",
    len(cs_dismissals_check) ==
    cs_batting_check["is_out"].eq(True).sum()
)

print(
    "Afghanistan dismissal flags complete:",
    afg_batting_check["is_out"].notna().all()
)

print(
    "Synthetic records correctly marked out:",
    cs_synthetic["is_out"].eq(True).all()
)

print("\nDISMISSAL RECONCILIATION COMPLETE — NO FILES MODIFIED")


CRICKETIQ — DISMISSAL RECONCILIATION

CRICSHEET
---------------------------------------------------------------------------
Source dismissal records: 37702
Unified batting involvement: 45483
Actual batting records: 45404
Synthetic dismissal records: 79
Dismissals marked out in unified data: 37702
Actual batting records marked out: 37623
Source dismissal key duplicates: 0

AFGHANISTAN
---------------------------------------------------------------------------
Afghanistan batting records: 2719
Marked out: 2280
Marked not out: 439
Missing dismissal flags: 0

FINAL CHECKS
---------------------------------------------------------------------------
Cricsheet dismissal reconciliation: True
Afghanistan dismissal flags complete: True
Synthetic records correctly marked out: True

DISMISSAL RECONCILIATION COMPLETE — NO FILES MODIFIED


In [24]:

import pandas as pd
import numpy as np

# Work on a separate copy
player_feature_work = unified_player_matches.copy()

player_feature_work["match_date"] = pd.to_datetime(
    player_feature_work["match_date"],
    errors="coerce"
)

# Ensure additive performance columns are numeric
numeric_columns = [
    "runs", "balls_faced", "fours", "sixes",
    "runs_conceded", "wickets", "legal_balls"
]

for column in numeric_columns:
    player_feature_work[column] = pd.to_numeric(
        player_feature_work[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# 1. CREATE DAILY PLAYER PERFORMANCE TOTALS
# ------------------------------------------------------------

daily_player_stats = (
    player_feature_work
    .groupby(
        ["cricketiq_player_id", "match_date"],
        as_index=False
    )
    .agg(
        matches_played=("match_key", "nunique"),

        batting_innings=("has_batting_involvement", "sum"),
        batting_runs=("runs", "sum"),
        batting_balls=("balls_faced", "sum"),
        batting_fours=("fours", "sum"),
        batting_sixes=("sixes", "sum"),
        batting_dismissals=(
            "is_out",
            lambda values: values.eq(True).sum()
        ),

        bowling_innings=("has_bowling_record", "sum"),
        bowling_runs_conceded=("runs_conceded", "sum"),
        bowling_wickets=("wickets", "sum"),
        bowling_legal_balls=("legal_balls", "sum")
    )
    .sort_values(
        ["cricketiq_player_id", "match_date"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. CALCULATE CUMULATIVE TOTALS BEFORE EACH DATE
# ------------------------------------------------------------

career_total_columns = [
    "matches_played",
    "batting_innings",
    "batting_runs",
    "batting_balls",
    "batting_fours",
    "batting_sixes",
    "batting_dismissals",
    "bowling_innings",
    "bowling_runs_conceded",
    "bowling_wickets",
    "bowling_legal_balls"
]

# Cumulative totals include the current date.
# Shift by one date to obtain only previous-date information.
prior_totals = (
    daily_player_stats
    .groupby("cricketiq_player_id")[career_total_columns]
    .cumsum()
    .groupby(daily_player_stats["cricketiq_player_id"])
    .shift(1)
    .fillna(0)
)

prior_totals.columns = [
    "pre_match_career_" + column
    for column in prior_totals.columns
]

daily_player_stats = pd.concat(
    [daily_player_stats, prior_totals],
    axis=1
)

# ------------------------------------------------------------
# 3. CALCULATE PRE-MATCH CAREER RATES
# ------------------------------------------------------------

def safe_divide(numerator, denominator):
    return numerator.div(denominator.where(denominator > 0))


daily_player_stats["pre_match_career_batting_average"] = safe_divide(
    daily_player_stats["pre_match_career_batting_runs"],
    daily_player_stats["pre_match_career_batting_dismissals"]
)

daily_player_stats["pre_match_career_batting_strike_rate"] = (
    safe_divide(
        daily_player_stats["pre_match_career_batting_runs"] * 100,
        daily_player_stats["pre_match_career_batting_balls"]
    )
)

daily_player_stats["pre_match_career_boundary_runs"] = (
    4 * daily_player_stats["pre_match_career_batting_fours"]
    + 6 * daily_player_stats["pre_match_career_batting_sixes"]
)

daily_player_stats["pre_match_career_boundary_percentage"] = (
    safe_divide(
        daily_player_stats["pre_match_career_boundary_runs"] * 100,
        daily_player_stats["pre_match_career_batting_runs"]
    )
)

daily_player_stats["pre_match_career_bowling_average"] = safe_divide(
    daily_player_stats["pre_match_career_bowling_runs_conceded"],
    daily_player_stats["pre_match_career_bowling_wickets"]
)

daily_player_stats["pre_match_career_bowling_economy"] = (
    safe_divide(
        daily_player_stats["pre_match_career_bowling_runs_conceded"] * 6,
        daily_player_stats["pre_match_career_bowling_legal_balls"]
    )
)

daily_player_stats["pre_match_career_bowling_strike_rate"] = (
    safe_divide(
        daily_player_stats["pre_match_career_bowling_legal_balls"],
        daily_player_stats["pre_match_career_bowling_wickets"]
    )
)

# ------------------------------------------------------------
# 4. MERGE PRE-MATCH FEATURES BACK TO PLAYER-MATCH ROWS
# ------------------------------------------------------------

pre_match_feature_columns = [
    column for column in daily_player_stats.columns
    if column.startswith("pre_match_")
]

player_match_features = player_feature_work.merge(
    daily_player_stats[
        ["cricketiq_player_id", "match_date"]
        + pre_match_feature_columns
    ],
    on=["cricketiq_player_id", "match_date"],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

print("=" * 75)
print("CRICKETIQ — PRE-MATCH CAREER FEATURE VALIDATION")
print("=" * 75)

print("\nOriginal player-match rows:",
      len(player_feature_work))

print("Feature dataset rows:",
      len(player_match_features))

print("New pre-match features:",
      len(pre_match_feature_columns))

print("Duplicate player-match keys:",
      player_match_features.duplicated(
          ["match_key", "cricketiq_player_id"]
      ).sum())

print("Missing pre-match feature values:",
      player_match_features[
          pre_match_feature_columns
      ].isna().sum().sum())

print("\nFirst 10 pre-match feature columns:")
print(pre_match_feature_columns[:10])

print("\nSample player career features:")
display(
    player_match_features[
        [
            "match_date",
            "canonical_player_name",
            "team",
            "pre_match_career_matches_played",
            "pre_match_career_batting_runs",
            "pre_match_career_batting_dismissals",
            "pre_match_career_batting_average",
            "pre_match_career_batting_strike_rate",
            "pre_match_career_bowling_wickets",
            "pre_match_career_bowling_average",
            "pre_match_career_bowling_economy"
        ]
    ].head(15)
)

print("\nPRE-MATCH CAREER FEATURES CREATED — NOT SAVED")


CRICKETIQ — PRE-MATCH CAREER FEATURE VALIDATION

Original player-match rows: 58319
Feature dataset rows: 58319
New pre-match features: 18
Duplicate player-match keys: 0
Missing pre-match feature values: 66272

First 10 pre-match feature columns:
['pre_match_career_matches_played', 'pre_match_career_batting_innings', 'pre_match_career_batting_runs', 'pre_match_career_batting_balls', 'pre_match_career_batting_fours', 'pre_match_career_batting_sixes', 'pre_match_career_batting_dismissals', 'pre_match_career_bowling_innings', 'pre_match_career_bowling_runs_conceded', 'pre_match_career_bowling_wickets']

Sample player career features:


,match_date,canonical_player_name,team,pre_match_career_matches_played,pre_match_career_batting_runs,pre_match_career_batting_dismissals,pre_match_career_batting_average,pre_match_career_batting_strike_rate,pre_match_career_bowling_wickets,pre_match_career_bowling_average,pre_match_career_bowling_economy
0,2002-06-27,Kumar Sangakkara,Sri Lanka,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
1,2002-06-27,Mahela Jayawardene,Sri Lanka,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
2,2002-06-27,RJ Kirtley,England,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
3,2002-06-27,CRD Fernando,Sri Lanka,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
4,2002-06-27,UDU Chandana,Sri Lanka,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
5,2002-06-27,GP Thorpe,England,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
6,2002-06-27,MS Atapattu,Sri Lanka,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
7,2002-06-27,MN Nawaz,Sri Lanka,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
8,2002-06-27,PD Collingwood,England,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
9,2002-06-27,AF Giles,England,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN



PRE-MATCH CAREER FEATURES CREATED — NOT SAVED


In [25]:

# ============================================================
# PRE-MATCH FEATURE MISSING-VALUE AUDIT
# ============================================================

print("=" * 75)
print("CRICKETIQ — PRE-MATCH FEATURE MISSING-VALUE AUDIT")
print("=" * 75)

feature_columns = [
    column for column in player_match_features.columns
    if column.startswith("pre_match_")
]

missing_report = pd.DataFrame({
    "missing_count": player_match_features[feature_columns].isna().sum(),
    "missing_percentage": (
        player_match_features[feature_columns].isna().mean() * 100
    ).round(2)
})

print("\nMissing values by feature:")
display(missing_report)

# Separate cumulative counts from calculated rates
rate_features = [
    column for column in feature_columns
    if any(term in column for term in [
        "average", "strike_rate", "economy", "percentage"
    ])
]

count_features = [
    column for column in feature_columns
    if column not in rate_features
]

print("\nCumulative count feature missing values:")
print(
    player_match_features[count_features]
    .isna()
    .sum()
    .sum()
)

print("\nCalculated rate feature missing values:")
print(
    player_match_features[rate_features]
    .isna()
    .sum()
    .sum()
)

print("\nCumulative count feature minimum values:")
display(
    player_match_features[count_features]
    .min()
    .to_frame("minimum")
)

# Check whether all count features are non-negative
negative_counts = (
    player_match_features[count_features] < 0
).sum()

print("\nNegative cumulative count values:")
display(negative_counts.to_frame("negative_values"))

# Check first recorded player-match observations
first_match_rows = (
    player_feature_work
    .sort_values(["match_date", "match_key"])
    .groupby("cricketiq_player_id", as_index=False)
    .first()
)

first_match_ids = first_match_rows[
    ["cricketiq_player_id", "match_date"]
]

first_match_check = player_match_features.merge(
    first_match_ids.assign(is_first_date=True),
    on=["cricketiq_player_id", "match_date"],
    how="left"
)

first_match_check = first_match_check[
    first_match_check["is_first_date"].eq(True)
]

print("\nFirst recorded date observations:")
print("Records:", len(first_match_check))

print(
    "Previous matches equal zero:",
    first_match_check[
        "pre_match_career_matches_played"
    ].eq(0).all()
)

print(
    "Previous batting runs equal zero:",
    first_match_check[
        "pre_match_career_batting_runs"
    ].eq(0).all()
)

print(
    "Previous bowling wickets equal zero:",
    first_match_check[
        "pre_match_career_bowling_wickets"
    ].eq(0).all()
)

print("\nMISSING-VALUE AUDIT COMPLETE — NO FILES MODIFIED")


CRICKETIQ — PRE-MATCH FEATURE MISSING-VALUE AUDIT

Missing values by feature:


,missing_count,missing_percentage
pre_match_career_matches_played,0,0.00
pre_match_career_batting_innings,0,0.00
pre_match_career_batting_runs,0,0.00
pre_match_career_batting_balls,0,0.00
pre_match_career_batting_fours,0,0.00
pre_match_career_batting_sixes,0,0.00
pre_match_career_batting_dismissals,0,0.00
pre_match_career_bowling_innings,0,0.00
pre_match_career_bowling_runs_conceded,0,0.00
pre_match_career_bowling_wickets,0,0.00



Cumulative count feature missing values:
0

Calculated rate feature missing values:
66272

Cumulative count feature minimum values:


,minimum
pre_match_career_matches_played,0.0
pre_match_career_batting_innings,0.0
pre_match_career_batting_runs,0.0
pre_match_career_batting_balls,0.0
pre_match_career_batting_fours,0.0
pre_match_career_batting_sixes,0.0
pre_match_career_batting_dismissals,0.0
pre_match_career_bowling_innings,0.0
pre_match_career_bowling_runs_conceded,0.0
pre_match_career_bowling_wickets,0.0



Negative cumulative count values:


,negative_values
pre_match_career_matches_played,0
pre_match_career_batting_innings,0
pre_match_career_batting_runs,0
pre_match_career_batting_balls,0
pre_match_career_batting_fours,0
pre_match_career_batting_sixes,0
pre_match_career_batting_dismissals,0
pre_match_career_bowling_innings,0
pre_match_career_bowling_runs_conceded,0
pre_match_career_bowling_wickets,0



First recorded date observations:
Records: 2070
Previous matches equal zero: True
Previous batting runs equal zero: True
Previous bowling wickets equal zero: True

MISSING-VALUE AUDIT COMPLETE — NO FILES MODIFIED


In [28]:

import pandas as pd
import numpy as np

def safe_rate(numerator, denominator):
    return numerator / denominator if denominator > 0 else np.nan


def build_recent_form_table(df, participation_column, mode):

    records = []

    active = df[df[participation_column].eq(True)].copy()

    active = active.sort_values(
        ["cricketiq_player_id", "match_date", "match_key"]
    )

    # Explicitly preserve column names, even for empty history.
    history_columns = active.columns.tolist()

    for player_id, player_df in active.groupby(
        "cricketiq_player_id", sort=False
    ):

        history = []

        for match_date, current_day in player_df.groupby(
            "match_date", sort=True
        ):

            current_day = current_day.sort_values("match_key")

            result = {
                "cricketiq_player_id": player_id,
                "match_date": match_date
            }

            for window in [5, 10]:

                recent = pd.DataFrame(
                    history[-window:],
                    columns=history_columns
                )

                result[f"pre_match_recent_{window}_{mode}_innings"] = (
                    len(recent)
                )

                if mode == "batting":

                    runs = recent["runs"].fillna(0).sum()
                    balls = recent["balls_faced"].fillna(0).sum()
                    fours = recent["fours"].fillna(0).sum()
                    sixes = recent["sixes"].fillna(0).sum()

                    dismissals = recent["is_out"].eq(True).sum()

                    boundary_runs = 4 * fours + 6 * sixes

                    result[
                        f"pre_match_recent_{window}_batting_runs"
                    ] = runs

                    result[
                        f"pre_match_recent_{window}_batting_dismissals"
                    ] = dismissals

                    result[
                        f"pre_match_recent_{window}_batting_average"
                    ] = safe_rate(runs, dismissals)

                    result[
                        f"pre_match_recent_{window}_batting_strike_rate"
                    ] = safe_rate(runs * 100, balls)

                    result[
                        f"pre_match_recent_{window}_boundary_percentage"
                    ] = safe_rate(boundary_runs * 100, runs)

                else:

                    runs = recent["runs_conceded"].fillna(0).sum()
                    wickets = recent["wickets"].fillna(0).sum()
                    legal_balls = recent["legal_balls"].fillna(0).sum()

                    result[
                        f"pre_match_recent_{window}_bowling_runs_conceded"
                    ] = runs

                    result[
                        f"pre_match_recent_{window}_bowling_wickets"
                    ] = wickets

                    result[
                        f"pre_match_recent_{window}_bowling_average"
                    ] = safe_rate(runs, wickets)

                    result[
                        f"pre_match_recent_{window}_bowling_economy"
                    ] = safe_rate(runs * 6, legal_balls)

                    result[
                        f"pre_match_recent_{window}_bowling_strike_rate"
                    ] = safe_rate(legal_balls, wickets)

            records.append(result)

            # Add current-day performances only after calculating
            # the pre-match features for that date.
            history.extend(current_day.to_dict("records"))

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 1. BUILD BATTING RECENT-FORM FEATURES
# ------------------------------------------------------------

recent_batting_features = build_recent_form_table(
    player_feature_work,
    participation_column="has_batting_involvement",
    mode="batting"
)

# ------------------------------------------------------------
# 2. BUILD BOWLING RECENT-FORM FEATURES
# ------------------------------------------------------------

recent_bowling_features = build_recent_form_table(
    player_feature_work,
    participation_column="has_bowling_record",
    mode="bowling"
)

# ------------------------------------------------------------
# 3. MERGE RECENT-FORM FEATURES
# ------------------------------------------------------------

player_match_features = player_match_features.merge(
    recent_batting_features,
    on=["cricketiq_player_id", "match_date"],
    how="left",
    validate="many_to_one"
)

player_match_features = player_match_features.merge(
    recent_bowling_features,
    on=["cricketiq_player_id", "match_date"],
    how="left",
    validate="many_to_one"
)

# No previous innings means zero recent innings.
# Undefined rates remain NaN.
recent_count_columns = [
    column for column in player_match_features.columns
    if column.startswith("pre_match_recent_")
    and column.endswith("_innings")
]

player_match_features[recent_count_columns] = (
    player_match_features[recent_count_columns].fillna(0)
)

# ------------------------------------------------------------
# 4. VALIDATION
# ------------------------------------------------------------

recent_columns = [
    column for column in player_match_features.columns
    if column.startswith("pre_match_recent_")
]

print("=" * 75)
print("CRICKETIQ — RECENT-FORM FEATURE VALIDATION")
print("=" * 75)

print("Original rows:", len(player_feature_work))
print("Feature rows:", len(player_match_features))
print("Recent-form features:", len(recent_columns))

print(
    "Duplicate player-match keys:",
    player_match_features.duplicated(
        ["match_key", "cricketiq_player_id"]
    ).sum()
)

print("\nRecent-form missing values:")
display(
    player_match_features[recent_columns]
    .isna()
    .sum()
    .to_frame("missing_values")
)

print("\nSample recent-form features:")
display(
    player_match_features[
        [
            "match_date",
            "canonical_player_name",
            "pre_match_recent_5_batting_innings",
            "pre_match_recent_5_batting_runs",
            "pre_match_recent_5_batting_average",
            "pre_match_recent_5_batting_strike_rate",
            "pre_match_recent_5_bowling_innings",
            "pre_match_recent_5_bowling_wickets",
            "pre_match_recent_5_bowling_economy"
        ]
    ].head(15)
)

print("\nRECENT-FORM FEATURES CREATED — NOT SAVED")


CRICKETIQ — RECENT-FORM FEATURE VALIDATION
Original rows: 58319
Feature rows: 58319
Recent-form features: 24
Duplicate player-match keys: 0

Recent-form missing values:


,missing_values
pre_match_recent_5_batting_innings,0
pre_match_recent_5_batting_runs,10117
pre_match_recent_5_batting_dismissals,10117
pre_match_recent_5_batting_average,12762
pre_match_recent_5_batting_strike_rate,12129
pre_match_recent_5_boundary_percentage,12428
pre_match_recent_10_batting_innings,0
pre_match_recent_10_batting_runs,10117
pre_match_recent_10_batting_dismissals,10117
pre_match_recent_10_batting_average,12712



Sample recent-form features:


,match_date,canonical_player_name,pre_match_recent_5_batting_innings,pre_match_recent_5_batting_runs,pre_match_recent_5_batting_average,pre_match_recent_5_batting_strike_rate,pre_match_recent_5_bowling_innings,pre_match_recent_5_bowling_wickets,pre_match_recent_5_bowling_economy
0,2002-06-27,Kumar Sangakkara,0.0,0.0,NaN,NaN,0.0,NaN,NaN
1,2002-06-27,Mahela Jayawardene,0.0,0.0,NaN,NaN,0.0,NaN,NaN
2,2002-06-27,RJ Kirtley,0.0,NaN,NaN,NaN,0.0,0.0,NaN
3,2002-06-27,CRD Fernando,0.0,0.0,NaN,NaN,0.0,0.0,NaN
4,2002-06-27,UDU Chandana,0.0,0.0,NaN,NaN,0.0,0.0,NaN
5,2002-06-27,GP Thorpe,0.0,0.0,NaN,NaN,0.0,NaN,NaN
6,2002-06-27,MS Atapattu,0.0,0.0,NaN,NaN,0.0,NaN,NaN
7,2002-06-27,MN Nawaz,0.0,0.0,NaN,NaN,0.0,NaN,NaN
8,2002-06-27,PD Collingwood,0.0,0.0,NaN,NaN,0.0,0.0,NaN
9,2002-06-27,AF Giles,0.0,NaN,NaN,NaN,0.0,0.0,NaN



RECENT-FORM FEATURES CREATED — NOT SAVED


In [29]:

import pandas as pd
import numpy as np

def safe_rate(numerator, denominator):
    return numerator / denominator if denominator > 0 else np.nan


def build_recent_form_table(df, participation_column, mode):

    records = []

    # Include every player-match row, not just active batting/bowling rows.
    working = df.sort_values(
        ["cricketiq_player_id", "match_date", "match_key"]
    ).copy()

    history_columns = working.columns.tolist()

    for player_id, player_df in working.groupby(
        "cricketiq_player_id", sort=False
    ):

        history = []

        for match_date, current_day in player_df.groupby(
            "match_date", sort=True
        ):

            current_day = current_day.sort_values("match_key")

            result = {
                "cricketiq_player_id": player_id,
                "match_date": match_date
            }

            # Calculate features using previous innings only.
            for window in [5, 10]:

                recent = pd.DataFrame(
                    history[-window:],
                    columns=history_columns
                )

                result[
                    f"pre_match_recent_{window}_{mode}_innings"
                ] = len(recent)

                if mode == "batting":

                    runs = recent["runs"].fillna(0).sum()
                    balls = recent["balls_faced"].fillna(0).sum()
                    fours = recent["fours"].fillna(0).sum()
                    sixes = recent["sixes"].fillna(0).sum()

                    dismissals = recent["is_out"].eq(True).sum()

                    boundary_runs = 4 * fours + 6 * sixes

                    result[
                        f"pre_match_recent_{window}_batting_runs"
                    ] = runs

                    result[
                        f"pre_match_recent_{window}_batting_dismissals"
                    ] = dismissals

                    result[
                        f"pre_match_recent_{window}_batting_average"
                    ] = safe_rate(runs, dismissals)

                    result[
                        f"pre_match_recent_{window}_batting_strike_rate"
                    ] = safe_rate(runs * 100, balls)

                    result[
                        f"pre_match_recent_{window}_boundary_percentage"
                    ] = safe_rate(boundary_runs * 100, runs)

                else:

                    runs = recent["runs_conceded"].fillna(0).sum()
                    wickets = recent["wickets"].fillna(0).sum()
                    legal_balls = recent["legal_balls"].fillna(0).sum()

                    result[
                        f"pre_match_recent_{window}_bowling_runs_conceded"
                    ] = runs

                    result[
                        f"pre_match_recent_{window}_bowling_wickets"
                    ] = wickets

                    result[
                        f"pre_match_recent_{window}_bowling_average"
                    ] = safe_rate(runs, wickets)

                    result[
                        f"pre_match_recent_{window}_bowling_economy"
                    ] = safe_rate(runs * 6, legal_balls)

                    result[
                        f"pre_match_recent_{window}_bowling_strike_rate"
                    ] = safe_rate(legal_balls, wickets)

            records.append(result)

            # Add only actual performances in this discipline to history.
            current_active = current_day[
                current_day[participation_column].eq(True)
            ]

            history.extend(current_active.to_dict("records"))

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 1. REBUILD BATTING AND BOWLING RECENT-FORM TABLES
# ------------------------------------------------------------

recent_batting_features = build_recent_form_table(
    player_feature_work,
    participation_column="has_batting_involvement",
    mode="batting"
)

recent_bowling_features = build_recent_form_table(
    player_feature_work,
    participation_column="has_bowling_record",
    mode="bowling"
)


# ------------------------------------------------------------
# 2. REMOVE OLD RECENT-FORM COLUMNS
# ------------------------------------------------------------

old_recent_columns = [
    column for column in player_match_features.columns
    if column.startswith("pre_match_recent_")
]

player_match_features = player_match_features.drop(
    columns=old_recent_columns
)


# ------------------------------------------------------------
# 3. MERGE THE CORRECTED FEATURES
# ------------------------------------------------------------

player_match_features = player_match_features.merge(
    recent_batting_features,
    on=["cricketiq_player_id", "match_date"],
    how="left",
    validate="many_to_one"
)

player_match_features = player_match_features.merge(
    recent_bowling_features,
    on=["cricketiq_player_id", "match_date"],
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 4. VALIDATION
# ------------------------------------------------------------

recent_columns = [
    column for column in player_match_features.columns
    if column.startswith("pre_match_recent_")
]

recent_count_columns = [
    column for column in recent_columns
    if column.endswith("_innings")
]

print("=" * 75)
print("CRICKETIQ — CORRECTED RECENT-FORM VALIDATION")
print("=" * 75)

print("Original rows:", len(player_feature_work))
print("Feature rows:", len(player_match_features))
print("Recent-form features:", len(recent_columns))

print(
    "Duplicate player-match keys:",
    player_match_features.duplicated(
        ["match_key", "cricketiq_player_id"]
    ).sum()
)

print(
    "\nMissing recent-form counts:",
    player_match_features[recent_count_columns].isna().sum().sum()
)

print("\nMissing values by feature:")
display(
    player_match_features[recent_columns]
    .isna()
    .sum()
    .to_frame("missing_values")
)

print("\nSample recent-form features:")
display(
    player_match_features[
        [
            "match_date",
            "canonical_player_name",
            "pre_match_recent_5_batting_innings",
            "pre_match_recent_5_batting_runs",
            "pre_match_recent_5_batting_average",
            "pre_match_recent_5_bowling_innings",
            "pre_match_recent_5_bowling_wickets",
            "pre_match_recent_5_bowling_economy"
        ]
    ].head(15)
)

print("\nCORRECTED RECENT-FORM FEATURES CREATED — NOT SAVED")


CRICKETIQ — CORRECTED RECENT-FORM VALIDATION
Original rows: 58319
Feature rows: 58319
Recent-form features: 24
Duplicate player-match keys: 0

Missing recent-form counts: 0

Missing values by feature:


,missing_values
pre_match_recent_5_batting_innings,0
pre_match_recent_5_batting_runs,0
pre_match_recent_5_batting_dismissals,0
pre_match_recent_5_batting_average,3980
pre_match_recent_5_batting_strike_rate,2859
pre_match_recent_5_boundary_percentage,3360
pre_match_recent_10_batting_innings,0
pre_match_recent_10_batting_runs,0
pre_match_recent_10_batting_dismissals,0
pre_match_recent_10_batting_average,3894



Sample recent-form features:


,match_date,canonical_player_name,pre_match_recent_5_batting_innings,pre_match_recent_5_batting_runs,pre_match_recent_5_batting_average,pre_match_recent_5_bowling_innings,pre_match_recent_5_bowling_wickets,pre_match_recent_5_bowling_economy
0,2002-06-27,Kumar Sangakkara,0,0.0,NaN,0,0.0,NaN
1,2002-06-27,Mahela Jayawardene,0,0.0,NaN,0,0.0,NaN
2,2002-06-27,RJ Kirtley,0,0.0,NaN,0,0.0,NaN
3,2002-06-27,CRD Fernando,0,0.0,NaN,0,0.0,NaN
4,2002-06-27,UDU Chandana,0,0.0,NaN,0,0.0,NaN
5,2002-06-27,GP Thorpe,0,0.0,NaN,0,0.0,NaN
6,2002-06-27,MS Atapattu,0,0.0,NaN,0,0.0,NaN
7,2002-06-27,MN Nawaz,0,0.0,NaN,0,0.0,NaN
8,2002-06-27,PD Collingwood,0,0.0,NaN,0,0.0,NaN
9,2002-06-27,AF Giles,0,0.0,NaN,0,0.0,NaN



CORRECTED RECENT-FORM FEATURES CREATED — NOT SAVED


In [30]:

# ------------------------------------------------------------
# CRICKETIQ — RECENT-FORM TEMPORAL LEAKAGE AUDIT
# ------------------------------------------------------------

def audit_recent_history(mode, participation_column, player_id_column):

    active = player_feature_work[
        player_feature_work[participation_column].eq(True)
    ].copy()

    # Select a player with sufficient historical performances.
    counts = active.groupby("cricketiq_player_id").size()
    eligible = counts[counts >= 11]

    if eligible.empty:
        print(f"No player has 11 {mode} performances.")
        return

    player_id = eligible.idxmax()

    player_history = active[
        active["cricketiq_player_id"].eq(player_id)
    ].sort_values(["match_date", "match_key"])

    # Select a date with at least 10 earlier performances.
    all_player_dates = player_feature_work[
        player_feature_work["cricketiq_player_id"].eq(player_id)
    ]["match_date"].drop_duplicates().sort_values()

    target_date = None

    for date in all_player_dates:
        previous = player_history[
            player_history["match_date"] < date
        ]

        if len(previous) >= 10:
            target_date = date
            break

    if target_date is None:
        print(f"Could not find a suitable {mode} audit date.")
        return

    previous = player_history[
        player_history["match_date"] < target_date
    ]

    feature_row = player_match_features[
        player_match_features["cricketiq_player_id"].eq(player_id)
        & player_match_features["match_date"].eq(target_date)
    ].iloc[0]

    print(f"\n{mode.upper()} AUDIT")
    print("Player:", feature_row["canonical_player_name"])
    print("Target match date:", target_date)
    print("Previous performances available:", len(previous))

    for window in [5, 10]:

        recent = previous.tail(window)

        if mode == "batting":

            expected_runs = recent["runs"].fillna(0).sum()
            expected_count = len(recent)
            expected_dismissals = recent["is_out"].eq(True).sum()

            actual_runs = feature_row[
                f"pre_match_recent_{window}_batting_runs"
            ]

            actual_count = feature_row[
                f"pre_match_recent_{window}_batting_innings"
            ]

            actual_dismissals = feature_row[
                f"pre_match_recent_{window}_batting_dismissals"
            ]

            checks = {
                "innings count": actual_count == expected_count,
                "runs": np.isclose(actual_runs, expected_runs),
                "dismissals": actual_dismissals == expected_dismissals
            }

        else:

            expected_runs = recent["runs_conceded"].fillna(0).sum()
            expected_count = len(recent)
            expected_wickets = recent["wickets"].fillna(0).sum()

            actual_runs = feature_row[
                f"pre_match_recent_{window}_bowling_runs_conceded"
            ]

            actual_count = feature_row[
                f"pre_match_recent_{window}_bowling_innings"
            ]

            actual_wickets = feature_row[
                f"pre_match_recent_{window}_bowling_wickets"
            ]

            checks = {
                "innings count": actual_count == expected_count,
                "runs conceded": np.isclose(actual_runs, expected_runs),
                "wickets": actual_wickets == expected_wickets
            }

        print(f"\nPrevious {window} {mode} innings:")
        print("Expected count:", expected_count)
        print("Actual count:", actual_count)
        print("Expected runs:", expected_runs)
        print("Actual runs:", actual_runs)

        if mode == "batting":
            print("Expected dismissals:", expected_dismissals)
            print("Actual dismissals:", actual_dismissals)
        else:
            print("Expected wickets:", expected_wickets)
            print("Actual wickets:", actual_wickets)

        print("Checks:", checks)
        print("Window passed:", all(checks.values()))


audit_recent_history(
    mode="batting",
    participation_column="has_batting_involvement",
    player_id_column="cricketiq_player_id"
)

audit_recent_history(
    mode="bowling",
    participation_column="has_bowling_record",
    player_id_column="cricketiq_player_id"
)




BATTING AUDIT
Player: Virat Kohli
Target match date: 2009-12-15 00:00:00
Previous performances available: 10

Previous 5 batting innings:
Expected count: 5
Actual count: 5
Expected runs: 137.0
Actual runs: 137.0
Expected dismissals: 3
Actual dismissals: 3
Checks: {'innings count': np.True_, 'runs': np.True_, 'dismissals': np.True_}
Window passed: True

Previous 10 batting innings:
Expected count: 10
Actual count: 10
Expected runs: 296.0
Actual runs: 296.0
Expected dismissals: 8
Actual dismissals: 8
Checks: {'innings count': np.True_, 'runs': np.True_, 'dismissals': np.True_}
Window passed: True

BOWLING AUDIT
Player: Shakib Al Hasan
Target match date: 2006-12-09 00:00:00
Previous performances available: 10

Previous 5 bowling innings:
Expected count: 5
Actual count: 5
Expected runs: 146.0
Actual runs: 146.0
Expected wickets: 7.0
Actual wickets: 7.0
Checks: {'innings count': np.True_, 'runs conceded': np.True_, 'wickets': np.True_}
Window passed: True

Previous 10 bowling innings:
Expe

In [31]:

# ------------------------------------------------------------
# CRICKETIQ — CONSOLIDATED PLAYER FEATURE QUALITY AUDIT
# ------------------------------------------------------------

player_pre_match_feature_columns = [
    column for column in player_match_features.columns
    if column.startswith("pre_match_career_")
    or column.startswith("pre_match_recent_")
]

# 1. Feature inventory
career_features = [
    c for c in player_pre_match_feature_columns
    if c.startswith("pre_match_career_")
]

recent_features = [
    c for c in player_pre_match_feature_columns
    if c.startswith("pre_match_recent_")
]

# 2. Numeric datatype validation
non_numeric_features = [
    c for c in player_pre_match_feature_columns
    if not pd.api.types.is_numeric_dtype(player_match_features[c])
]

# 3. Negative-value validation
negative_counts = {}

for column in player_pre_match_feature_columns:
    negative_counts[column] = (
        player_match_features[column].lt(0).sum()
    )

negative_violations = {
    column: count
    for column, count in negative_counts.items()
    if count > 0
}

# 4. Same-day snapshot consistency
# A player's pre-match features should be identical
# across all their matches on the same date.

same_day_variation = (
    player_match_features
    .groupby(
        ["cricketiq_player_id", "match_date"]
    )[player_pre_match_feature_columns]
    .nunique(dropna=False)
)

same_day_inconsistent = (
    same_day_variation.gt(1).any(axis=1).sum()
)

# 5. Duplicate player-match validation
duplicate_keys = player_match_features.duplicated(
    ["match_key", "cricketiq_player_id"]
).sum()

# 6. Missing feature count
missing_feature_values = (
    player_match_features[player_pre_match_feature_columns]
    .isna()
    .sum()
)

# 7. Final report
print("=" * 75)
print("CRICKETIQ — CONSOLIDATED PLAYER FEATURE QUALITY AUDIT")
print("=" * 75)

print("Total player-match records:", len(player_match_features))
print("Career features:", len(career_features))
print("Recent-form features:", len(recent_features))
print("Total pre-match features:", len(player_pre_match_feature_columns))

print("\nDATA TYPE CHECK")
print("Non-numeric features:", len(non_numeric_features))
print(non_numeric_features)

print("\nNEGATIVE VALUE CHECK")
print("Features with negative values:", len(negative_violations))
print(negative_violations)

print("\nSAME-DAY SNAPSHOT CHECK")
print("Inconsistent player-date snapshots:", same_day_inconsistent)

print("\nDUPLICATE CHECK")
print("Duplicate player-match keys:", duplicate_keys)

print("\nMISSING VALUE SUMMARY")
print("Features containing missing values:", (missing_feature_values > 0).sum())
print("Total missing feature values:", missing_feature_values.sum())

print("\nFINAL STATUS")
if (
    len(non_numeric_features) == 0
    and len(negative_violations) == 0
    and same_day_inconsistent == 0
    and duplicate_keys == 0
):
    print("STRUCTURAL QUALITY CHECKS PASSED")
else:
    print("REVIEW REQUIRED — CHECK THE RESULTS ABOVE")

print("\nFEATURE DATASET NOT SAVED")


CRICKETIQ — CONSOLIDATED PLAYER FEATURE QUALITY AUDIT
Total player-match records: 58319
Career features: 18
Recent-form features: 24
Total pre-match features: 42

DATA TYPE CHECK
Non-numeric features: 0
[]

NEGATIVE VALUE CHECK
Features with negative values: 0
{}

SAME-DAY SNAPSHOT CHECK
Inconsistent player-date snapshots: 0

DUPLICATE CHECK
Duplicate player-match keys: 0

MISSING VALUE SUMMARY
Features containing missing values: 18
Total missing feature values: 203176

FINAL STATUS
STRUCTURAL QUALITY CHECKS PASSED

FEATURE DATASET NOT SAVED


In [32]:

# ------------------------------------------------------------
# CRICKETIQ — MISSING VALUE SEMANTICS AUDIT
# ------------------------------------------------------------

rate_features = [
    c for c in player_pre_match_feature_columns
    if any(metric in c for metric in [
        "batting_average",
        "batting_strike_rate",
        "boundary_percentage",
        "bowling_average",
        "bowling_economy",
        "bowling_strike_rate"
    ])
]

non_rate_features = [
    c for c in player_pre_match_feature_columns
    if c not in rate_features
]

print("=" * 75)
print("CRICKETIQ — MISSING VALUE SEMANTICS AUDIT")
print("=" * 75)

print("Rate features:", len(rate_features))
print("Non-rate features:", len(non_rate_features))

print("\nMissing values in rate features:")
display(
    player_match_features[rate_features]
    .isna()
    .sum()
    .to_frame("missing_values")
)

non_rate_missing = (
    player_match_features[non_rate_features]
    .isna()
    .sum()
)

print("\nMissing values in non-rate features:")
print("Total:", non_rate_missing.sum())
print(non_rate_missing[non_rate_missing > 0])

# ------------------------------------------------------------
# CAREER RATE DENOMINATOR CHECKS
# ------------------------------------------------------------

career_denominator_checks = {
    "pre_match_career_batting_average":
        "pre_match_career_batting_dismissals",

    "pre_match_career_batting_strike_rate":
        "pre_match_career_batting_balls",

    "pre_match_career_boundary_percentage":
        "pre_match_career_batting_runs",

    "pre_match_career_bowling_average":
        "pre_match_career_bowling_wickets",

    "pre_match_career_bowling_economy":
        "pre_match_career_bowling_legal_balls",

    "pre_match_career_bowling_strike_rate":
        "pre_match_career_bowling_wickets"
}

print("\nCAREER RATE DENOMINATOR VALIDATION")

for rate_column, denominator_column in career_denominator_checks.items():

    expected_missing = (
        player_match_features[denominator_column].eq(0)
    )

    actual_missing = (
        player_match_features[rate_column].isna()
    )

    mismatches = (
        expected_missing != actual_missing
    ).sum()

    print(
        f"{rate_column}: "
        f"denominator mismatches = {mismatches}"
    )

print("\nFINAL MISSING VALUE STATUS")

if non_rate_missing.sum() == 0:
    print("PASS: No missing values in count or total features.")
else:
    print("REVIEW: Some count or total features contain missing values.")


CRICKETIQ — MISSING VALUE SEMANTICS AUDIT
Rate features: 18
Non-rate features: 24

Missing values in rate features:


,missing_values
pre_match_career_batting_average,3894
pre_match_career_batting_strike_rate,2859
pre_match_career_boundary_percentage,3342
pre_match_career_bowling_average,20295
pre_match_career_bowling_economy,15587
pre_match_career_bowling_strike_rate,20295
pre_match_recent_5_batting_average,3980
pre_match_recent_5_batting_strike_rate,2859
pre_match_recent_5_boundary_percentage,3360
pre_match_recent_10_batting_average,3894



Missing values in non-rate features:
Total: 0
Series([], dtype: int64)

CAREER RATE DENOMINATOR VALIDATION
pre_match_career_batting_average: denominator mismatches = 0
pre_match_career_batting_strike_rate: denominator mismatches = 0
pre_match_career_boundary_percentage: denominator mismatches = 0
pre_match_career_bowling_average: denominator mismatches = 0
pre_match_career_bowling_economy: denominator mismatches = 0
pre_match_career_bowling_strike_rate: denominator mismatches = 0

FINAL MISSING VALUE STATUS
PASS: No missing values in count or total features.


In [33]:

# ------------------------------------------------------------
# CRICKETIQ — RECENT-FORM RATE DENOMINATOR AUDIT
# ------------------------------------------------------------

recent_denominator_checks = {}

for window in [5, 10]:

    recent_denominator_checks.update({

        f"pre_match_recent_{window}_batting_average":
            f"pre_match_recent_{window}_batting_dismissals",

        f"pre_match_recent_{window}_batting_strike_rate":
            f"pre_match_recent_{window}_batting_balls",

        f"pre_match_recent_{window}_boundary_percentage":
            f"pre_match_recent_{window}_batting_runs",

        f"pre_match_recent_{window}_bowling_average":
            f"pre_match_recent_{window}_bowling_wickets",

        f"pre_match_recent_{window}_bowling_economy":
            f"pre_match_recent_{window}_bowling_legal_balls",

        f"pre_match_recent_{window}_bowling_strike_rate":
            f"pre_match_recent_{window}_bowling_wickets"
    })


# Check whether all required denominator columns exist.
missing_denominator_columns = [
    column for column in recent_denominator_checks.values()
    if column not in player_match_features.columns
]

print("=" * 75)
print("CRICKETIQ — RECENT-FORM RATE DENOMINATOR AUDIT")
print("=" * 75)

if missing_denominator_columns:

    print("Missing denominator columns:")
    print(missing_denominator_columns)

else:

    total_mismatches = 0

    for rate_column, denominator_column in recent_denominator_checks.items():

        expected_missing = (
            player_match_features[denominator_column].eq(0)
        )

        actual_missing = (
            player_match_features[rate_column].isna()
        )

        mismatches = (
            expected_missing != actual_missing
        ).sum()

        total_mismatches += mismatches

        print(
            f"{rate_column}: "
            f"denominator mismatches = {mismatches}"
        )

    print("\nTotal rate features checked:", len(recent_denominator_checks))
    print("Total denominator mismatches:", total_mismatches)

    if total_mismatches == 0:
        print("\nPASS: All recent-form missing values are mathematically justified.")
    else:
        print("\nREVIEW REQUIRED: Some recent-form rates need investigation.")


CRICKETIQ — RECENT-FORM RATE DENOMINATOR AUDIT
Missing denominator columns:
['pre_match_recent_5_batting_balls', 'pre_match_recent_5_bowling_legal_balls', 'pre_match_recent_10_batting_balls', 'pre_match_recent_10_bowling_legal_balls']


In [34]:

# ------------------------------------------------------------
# CRICKETIQ — REBUILD RECENT-FORM FEATURES WITH DENOMINATORS
# ------------------------------------------------------------

def safe_rate(numerator, denominator):
    return numerator / denominator if denominator > 0 else np.nan


def build_recent_form_table(df, participation_column, mode):

    records = []

    working = df.sort_values(
        ["cricketiq_player_id", "match_date", "match_key"]
    ).copy()

    history_columns = working.columns.tolist()

    for player_id, player_df in working.groupby(
        "cricketiq_player_id", sort=False
    ):

        history = []

        for match_date, current_day in player_df.groupby(
            "match_date", sort=True
        ):

            current_day = current_day.sort_values("match_key")

            result = {
                "cricketiq_player_id": player_id,
                "match_date": match_date
            }

            for window in [5, 10]:

                recent = pd.DataFrame(
                    history[-window:],
                    columns=history_columns
                )

                result[
                    f"pre_match_recent_{window}_{mode}_innings"
                ] = len(recent)

                if mode == "batting":

                    runs = recent["runs"].fillna(0).sum()
                    balls = recent["balls_faced"].fillna(0).sum()
                    fours = recent["fours"].fillna(0).sum()
                    sixes = recent["sixes"].fillna(0).sum()

                    dismissals = recent["is_out"].eq(True).sum()

                    boundary_runs = 4 * fours + 6 * sixes

                    result[
                        f"pre_match_recent_{window}_batting_runs"
                    ] = runs

                    result[
                        f"pre_match_recent_{window}_batting_balls"
                    ] = balls

                    result[
                        f"pre_match_recent_{window}_batting_dismissals"
                    ] = dismissals

                    result[
                        f"pre_match_recent_{window}_batting_average"
                    ] = safe_rate(runs, dismissals)

                    result[
                        f"pre_match_recent_{window}_batting_strike_rate"
                    ] = safe_rate(runs * 100, balls)

                    result[
                        f"pre_match_recent_{window}_boundary_percentage"
                    ] = safe_rate(boundary_runs * 100, runs)

                else:

                    runs = recent["runs_conceded"].fillna(0).sum()
                    wickets = recent["wickets"].fillna(0).sum()
                    legal_balls = recent["legal_balls"].fillna(0).sum()

                    result[
                        f"pre_match_recent_{window}_bowling_runs_conceded"
                    ] = runs

                    result[
                        f"pre_match_recent_{window}_bowling_legal_balls"
                    ] = legal_balls

                    result[
                        f"pre_match_recent_{window}_bowling_wickets"
                    ] = wickets

                    result[
                        f"pre_match_recent_{window}_bowling_average"
                    ] = safe_rate(runs, wickets)

                    result[
                        f"pre_match_recent_{window}_bowling_economy"
                    ] = safe_rate(runs * 6, legal_balls)

                    result[
                        f"pre_match_recent_{window}_bowling_strike_rate"
                    ] = safe_rate(legal_balls, wickets)

            records.append(result)

            # Add only performances from the current discipline.
            current_active = current_day[
                current_day[participation_column].eq(True)
            ]

            history.extend(current_active.to_dict("records"))

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 1. REBUILD BOTH TABLES
# ------------------------------------------------------------

recent_batting_features = build_recent_form_table(
    player_feature_work,
    "has_batting_involvement",
    "batting"
)

recent_bowling_features = build_recent_form_table(
    player_feature_work,
    "has_bowling_record",
    "bowling"
)


# ------------------------------------------------------------
# 2. REMOVE PREVIOUS RECENT-FORM FEATURES
# ------------------------------------------------------------

old_recent_columns = [
    column for column in player_match_features.columns
    if column.startswith("pre_match_recent_")
]

player_match_features = player_match_features.drop(
    columns=old_recent_columns
)


# ------------------------------------------------------------
# 3. MERGE UPDATED FEATURES
# ------------------------------------------------------------

player_match_features = player_match_features.merge(
    recent_batting_features,
    on=["cricketiq_player_id", "match_date"],
    how="left",
    validate="many_to_one"
)

player_match_features = player_match_features.merge(
    recent_bowling_features,
    on=["cricketiq_player_id", "match_date"],
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 4. VALIDATE UPDATED STRUCTURE
# ------------------------------------------------------------

recent_columns = [
    c for c in player_match_features.columns
    if c.startswith("pre_match_recent_")
]

print("=" * 75)
print("CRICKETIQ — UPDATED RECENT-FORM FEATURE VALIDATION")
print("=" * 75)

print("Original rows:", len(player_feature_work))
print("Feature rows:", len(player_match_features))
print("Recent-form feature count:", len(recent_columns))

print(
    "Duplicate player-match keys:",
    player_match_features.duplicated(
        ["match_key", "cricketiq_player_id"]
    ).sum()
)

print("\nNew denominator columns:")
display(
    player_match_features[
        [
            "pre_match_recent_5_batting_balls",
            "pre_match_recent_10_batting_balls",
            "pre_match_recent_5_bowling_legal_balls",
            "pre_match_recent_10_bowling_legal_balls"
        ]
    ].describe()
)

print("\nUPDATED RECENT-FORM FEATURES CREATED — NOT SAVED")


CRICKETIQ — UPDATED RECENT-FORM FEATURE VALIDATION
Original rows: 58319
Feature rows: 58319
Recent-form feature count: 28
Duplicate player-match keys: 0

New denominator columns:


,pre_match_recent_5_batting_balls,pre_match_recent_10_batting_balls,pre_match_recent_5_bowling_legal_balls,pre_match_recent_10_bowling_legal_balls
count,58319.000000,58319.000000,58319.000000,58319.000000
mean,121.572404,228.407449,123.359231,226.435930
std,92.972107,173.972352,106.184300,208.343245
min,0.000000,0.000000,0.000000,0.000000
25%,44.000000,77.000000,0.000000,0.000000
50%,104.000000,202.000000,114.000000,186.000000
75%,182.000000,357.000000,228.000000,445.000000
max,591.000000,957.000000,300.000000,600.000000



UPDATED RECENT-FORM FEATURES CREATED — NOT SAVED


In [35]:

# ------------------------------------------------------------
# CRICKETIQ — RECENT-FORM RATE DENOMINATOR AUDIT
# ------------------------------------------------------------

recent_denominator_checks = {}

for window in [5, 10]:

    recent_denominator_checks.update({

        f"pre_match_recent_{window}_batting_average":
            f"pre_match_recent_{window}_batting_dismissals",

        f"pre_match_recent_{window}_batting_strike_rate":
            f"pre_match_recent_{window}_batting_balls",

        f"pre_match_recent_{window}_boundary_percentage":
            f"pre_match_recent_{window}_batting_runs",

        f"pre_match_recent_{window}_bowling_average":
            f"pre_match_recent_{window}_bowling_wickets",

        f"pre_match_recent_{window}_bowling_economy":
            f"pre_match_recent_{window}_bowling_legal_balls",

        f"pre_match_recent_{window}_bowling_strike_rate":
            f"pre_match_recent_{window}_bowling_wickets"
    })


# Check whether all required denominator columns exist.
missing_denominator_columns = [
    column for column in recent_denominator_checks.values()
    if column not in player_match_features.columns
]

print("=" * 75)
print("CRICKETIQ — RECENT-FORM RATE DENOMINATOR AUDIT")
print("=" * 75)

if missing_denominator_columns:

    print("Missing denominator columns:")
    print(missing_denominator_columns)

else:

    total_mismatches = 0

    for rate_column, denominator_column in recent_denominator_checks.items():

        expected_missing = (
            player_match_features[denominator_column].eq(0)
        )

        actual_missing = (
            player_match_features[rate_column].isna()
        )

        mismatches = (
            expected_missing != actual_missing
        ).sum()

        total_mismatches += mismatches

        print(
            f"{rate_column}: "
            f"denominator mismatches = {mismatches}"
        )

    print("\nTotal rate features checked:", len(recent_denominator_checks))
    print("Total denominator mismatches:", total_mismatches)

    if total_mismatches == 0:
        print("\nPASS: All recent-form missing values are mathematically justified.")
    else:
        print("\nREVIEW REQUIRED: Some recent-form rates need investigation.")


CRICKETIQ — RECENT-FORM RATE DENOMINATOR AUDIT
pre_match_recent_5_batting_average: denominator mismatches = 0
pre_match_recent_5_batting_strike_rate: denominator mismatches = 0
pre_match_recent_5_boundary_percentage: denominator mismatches = 0
pre_match_recent_5_bowling_average: denominator mismatches = 0
pre_match_recent_5_bowling_economy: denominator mismatches = 0
pre_match_recent_5_bowling_strike_rate: denominator mismatches = 0
pre_match_recent_10_batting_average: denominator mismatches = 0
pre_match_recent_10_batting_strike_rate: denominator mismatches = 0
pre_match_recent_10_boundary_percentage: denominator mismatches = 0
pre_match_recent_10_bowling_average: denominator mismatches = 0
pre_match_recent_10_bowling_economy: denominator mismatches = 0
pre_match_recent_10_bowling_strike_rate: denominator mismatches = 0

Total rate features checked: 12
Total denominator mismatches: 0

PASS: All recent-form missing values are mathematically justified.


In [36]:

# ------------------------------------------------------------
# CRICKETIQ — FINAL 46-FEATURE VALIDATION
# ------------------------------------------------------------

# Refresh feature list after adding the four new columns.
player_pre_match_feature_columns = [
    c for c in player_match_features.columns
    if c.startswith("pre_match_career_")
    or c.startswith("pre_match_recent_")
]

career_features = [
    c for c in player_pre_match_feature_columns
    if c.startswith("pre_match_career_")
]

recent_features = [
    c for c in player_pre_match_feature_columns
    if c.startswith("pre_match_recent_")
]

# ------------------------------------------------------------
# 1. STRUCTURAL VALIDATION
# ------------------------------------------------------------

non_numeric = [
    c for c in player_pre_match_feature_columns
    if not pd.api.types.is_numeric_dtype(player_match_features[c])
]

negative_features = {
    c: int(player_match_features[c].lt(0).sum())
    for c in player_pre_match_feature_columns
    if player_match_features[c].lt(0).any()
}

duplicate_keys = player_match_features.duplicated(
    ["match_key", "cricketiq_player_id"]
).sum()

same_day_variation = (
    player_match_features
    .groupby(["cricketiq_player_id", "match_date"])
    [player_pre_match_feature_columns]
    .nunique(dropna=False)
)

same_day_inconsistent = (
    same_day_variation.gt(1).any(axis=1).sum()
)

# ------------------------------------------------------------
# 2. MISSING VALUE VALIDATION
# ------------------------------------------------------------

rate_features = [
    c for c in player_pre_match_feature_columns
    if any(metric in c for metric in [
        "batting_average",
        "batting_strike_rate",
        "boundary_percentage",
        "bowling_average",
        "bowling_economy",
        "bowling_strike_rate"
    ])
]

non_rate_features = [
    c for c in player_pre_match_feature_columns
    if c not in rate_features
]

non_rate_missing = (
    player_match_features[non_rate_features].isna().sum().sum()
)

denominator_checks = {}

for window in [None, 5, 10]:

    prefix = "pre_match_career" if window is None else f"pre_match_recent_{window}"

    if window is None:
        mapping = {
            f"{prefix}_batting_average":
                f"{prefix}_batting_dismissals",
            f"{prefix}_batting_strike_rate":
                f"{prefix}_batting_balls",
            f"{prefix}_boundary_percentage":
                f"{prefix}_batting_runs",
            f"{prefix}_bowling_average":
                f"{prefix}_bowling_wickets",
            f"{prefix}_bowling_economy":
                f"{prefix}_bowling_legal_balls",
            f"{prefix}_bowling_strike_rate":
                f"{prefix}_bowling_wickets"
        }
    else:
        mapping = {
            f"{prefix}_batting_average":
                f"{prefix}_batting_dismissals",
            f"{prefix}_batting_strike_rate":
                f"{prefix}_batting_balls",
            f"{prefix}_boundary_percentage":
                f"{prefix}_batting_runs",
            f"{prefix}_bowling_average":
                f"{prefix}_bowling_wickets",
            f"{prefix}_bowling_economy":
                f"{prefix}_bowling_legal_balls",
            f"{prefix}_bowling_strike_rate":
                f"{prefix}_bowling_wickets"
        }

    denominator_checks.update(mapping)

denominator_mismatches = {}

for rate_column, denominator_column in denominator_checks.items():

    expected_missing = (
        player_match_features[denominator_column].eq(0)
    )

    actual_missing = (
        player_match_features[rate_column].isna()
    )

    mismatches = int(
        (expected_missing != actual_missing).sum()
    )

    denominator_mismatches[rate_column] = mismatches

# ------------------------------------------------------------
# 3. INDEPENDENT TEMPORAL SPOT CHECK
# ------------------------------------------------------------

def final_temporal_audit(mode, participation_column):

    active = player_feature_work[
        player_feature_work[participation_column].eq(True)
    ].copy()

    counts = active.groupby("cricketiq_player_id").size()
    eligible = counts[counts >= 11]

    if eligible.empty:
        return {"mode": mode, "passed": False, "reason": "No eligible player"}

    player_id = eligible.idxmax()

    history = active[
        active["cricketiq_player_id"].eq(player_id)
    ].sort_values(["match_date", "match_key"])

    all_dates = player_feature_work[
        player_feature_work["cricketiq_player_id"].eq(player_id)
    ]["match_date"].drop_duplicates().sort_values()

    target_date = None

    for date in all_dates:
        previous = history[history["match_date"] < date]
        if len(previous) >= 10:
            target_date = date
            break

    if target_date is None:
        return {"mode": mode, "passed": False, "reason": "No audit date"}

    previous = history[history["match_date"] < target_date]

    feature_row = player_match_features[
        player_match_features["cricketiq_player_id"].eq(player_id)
        & player_match_features["match_date"].eq(target_date)
    ].iloc[0]

    checks = {}

    for window in [5, 10]:

        recent = previous.tail(window)

        if mode == "batting":

            expected = {
                "innings": len(recent),
                "runs": recent["runs"].fillna(0).sum(),
                "balls": recent["balls_faced"].fillna(0).sum(),
                "dismissals": recent["is_out"].eq(True).sum()
            }

            actual = {
                "innings": feature_row[
                    f"pre_match_recent_{window}_batting_innings"
                ],
                "runs": feature_row[
                    f"pre_match_recent_{window}_batting_runs"
                ],
                "balls": feature_row[
                    f"pre_match_recent_{window}_batting_balls"
                ],
                "dismissals": feature_row[
                    f"pre_match_recent_{window}_batting_dismissals"
                ]
            }

        else:

            expected = {
                "innings": len(recent),
                "runs_conceded": recent["runs_conceded"].fillna(0).sum(),
                "legal_balls": recent["legal_balls"].fillna(0).sum(),
                "wickets": recent["wickets"].fillna(0).sum()
            }

            actual = {
                "innings": feature_row[
                    f"pre_match_recent_{window}_bowling_innings"
                ],
                "runs_conceded": feature_row[
                    f"pre_match_recent_{window}_bowling_runs_conceded"
                ],
                "legal_balls": feature_row[
                    f"pre_match_recent_{window}_bowling_legal_balls"
                ],
                "wickets": feature_row[
                    f"pre_match_recent_{window}_bowling_wickets"
                ]
            }

        for metric in expected:
            checks[f"{window}_{metric}"] = np.isclose(
                expected[metric], actual[metric]
            )

    return {
        "mode": mode,
        "player": feature_row["canonical_player_name"],
        "target_date": target_date,
        "checks": checks,
        "passed": all(checks.values())
    }


batting_temporal = final_temporal_audit(
    "batting", "has_batting_involvement"
)

bowling_temporal = final_temporal_audit(
    "bowling", "has_bowling_record"
)

# ------------------------------------------------------------
# 4. FINAL REPORT
# ------------------------------------------------------------

print("=" * 75)
print("CRICKETIQ — FINAL PLAYER FEATURE VALIDATION")
print("=" * 75)

print("\nFEATURE INVENTORY")
print("Player-match rows:", len(player_match_features))
print("Career features:", len(career_features))
print("Recent-form features:", len(recent_features))
print("Total features:", len(player_pre_match_feature_columns))

print("\nSTRUCTURAL CHECKS")
print("Non-numeric features:", len(non_numeric))
print("Negative-value features:", len(negative_features))
print("Duplicate player-match keys:", duplicate_keys)
print("Inconsistent same-day snapshots:", same_day_inconsistent)

print("\nMISSING VALUE CHECKS")
print("Rate features:", len(rate_features))
print("Non-rate features:", len(non_rate_features))
print("Missing values in non-rate features:", non_rate_missing)
print("Rate denominator mismatches:", sum(denominator_mismatches.values()))

print("\nTEMPORAL VALIDATION")
print("Batting audit:", batting_temporal)
print("Bowling audit:", bowling_temporal)

all_passed = (
    len(player_pre_match_feature_columns) == 46
    and len(career_features) == 18
    and len(recent_features) == 28
    and len(non_numeric) == 0
    and len(negative_features) == 0
    and duplicate_keys == 0
    and same_day_inconsistent == 0
    and non_rate_missing == 0
    and sum(denominator_mismatches.values()) == 0
    and batting_temporal["passed"]
    and bowling_temporal["passed"]
)

print("\n" + "=" * 75)

if all_passed:
    print("FINAL RESULT: ALL CHECKS PASSED")
    print("46 PRE-MATCH PLAYER FEATURES ARE VALIDATED.")
else:
    print("FINAL RESULT: REVIEW REQUIRED")
    print("Inspect the failed checks before publication.")

print("\nDATASET NOT SAVED")



CRICKETIQ — FINAL PLAYER FEATURE VALIDATION

FEATURE INVENTORY
Player-match rows: 58319
Career features: 18
Recent-form features: 28
Total features: 46

STRUCTURAL CHECKS
Non-numeric features: 0
Negative-value features: 0
Duplicate player-match keys: 0
Inconsistent same-day snapshots: 0

MISSING VALUE CHECKS
Rate features: 18
Non-rate features: 28
Missing values in non-rate features: 0
Rate denominator mismatches: 0

TEMPORAL VALIDATION
Batting audit: {'mode': 'batting', 'player': 'Virat Kohli', 'target_date': Timestamp('2009-12-15 00:00:00'), 'checks': {'5_innings': np.True_, '5_runs': np.True_, '5_balls': np.True_, '5_dismissals': np.True_, '10_innings': np.True_, '10_runs': np.True_, '10_balls': np.True_, '10_dismissals': np.True_}, 'passed': True}
Bowling audit: {'mode': 'bowling', 'player': 'Shakib Al Hasan', 'target_date': Timestamp('2006-12-09 00:00:00'), 'checks': {'5_innings': np.True_, '5_runs_conceded': np.True_, '5_legal_balls': np.True_, '5_wickets': np.True_, '10_innings'

In [37]:

from pathlib import Path

# Save Notebook 07 output separately from canonical source files.

output_dir = Path(r"C:\CricIQ\data\features\player_data")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "cricketiq_player_match_features.csv"

player_match_features.to_csv(output_path, index=False)

# Verify the saved dataset.
saved_features = pd.read_csv(output_path)

print("NOTEBOOK 07 — DATASET PUBLICATION CHECK")
print("Saved path:", output_path)
print("Original shape:", player_match_features.shape)
print("Reloaded shape:", saved_features.shape)
print(
    "Duplicate player-match keys:",
    saved_features.duplicated(
        ["match_key", "cricketiq_player_id"]
    ).sum()
)

print("\nDATASET SAVED AND RELOADED SUCCESSFULLY")


NOTEBOOK 07 — DATASET PUBLICATION CHECK
Saved path: C:\CricIQ\data\features\player_data\cricketiq_player_match_features.csv
Original shape: (58319, 72)
Reloaded shape: (58319, 72)
Duplicate player-match keys: 0

DATASET SAVED AND RELOADED SUCCESSFULLY


C:\Users\ASUS\AppData\Local\Temp\ipykernel_12484\2054255835.py:13: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  saved_features = pd.read_csv(output_path)
